# NDI Task 3: independent self-report target (next-day mood and stress), on the v1.4 data

Revision analyses memo, Task 3; wellness rules W1-W4 from the v1.4 memo; changes from the **v1.4 review memo** (Sections 1-4). Run `NDI_sleep_score_check.ipynb` first: T3-0 stops unless its verdict is PASS. Then commit this notebook and `config_task3_v1_4.json`, tag `v1.4-task3`, and **Runtime > Disconnect and delete runtime, then Run all** (about 1 hour). G-0 asks once for the read-only token.

- Data cells are the v1.4 cells, unchanged (CELL 1.0, CELL 2, CELL 3, CELL 4, CELL 5, CELL 6, D-2, W-1, CELL 6.5, CELL 14.4, CELL 14.10). V12-1 and V12-2 are unchanged from v1.4.
- **T3-0**: v1.4 settings checked against tag `v1.4`; sleep-score verdict must be PASS; `config_task3_v1_4.json` (with the persistence references, fixed before the run); stops unless tagged.
- **T3-1** Task 3 targets behind flags; **T3-R** v1.4 reproduced exactly (first, and last in T3-H); **T3-2** runs, eligibility, complete-pairs check; **T3-3** statistics (artifact bound on the same participants); **T3-4** without p12 (sensitivity); **T3-P** persistence references; **T3-7** reporting items from v1.4 (target-swap without p12, Figure 4 from the L2-removed state, Table 4, corrected Table 2); **T3-5** Figure 2; **T3-6** independent check; **T3-H** notes and hand-in check.

Writes only to `results_v1.4_task3/`. `results_v1.2`, `results_v1.3`, `results_v1.4` and the frozen v1 archive are read, never written.

## Data (v1.4 cells, unchanged)

In [ ]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))

In [ ]:
# ================================================================
# CELL 2  INSTALL DEPENDENCIES + IMPORTS
# ================================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["xgboost", "shap"]:
    install(pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

import importlib.util, sys as _sys

spec = importlib.util.spec_from_file_location("ndi_pipeline", PIPELINE_MODULE)
ndi  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ndi)

print("ndi_pipeline loaded. Components available:")
for fn in ["neutrosophic_decompose","blocked_tscv","run_baselines",
           "lambda_grid_search","shap_attribution"]:
    print(f"  ndi.{fn}: {'OK' if hasattr(ndi, fn) else 'MISSING'}")

In [ ]:
# ================================================================
# CELL 3 -- LOAD DEPRESJON DATASET
# Structure: depresjon-dataset/data/condition/condition_1.csv ...
#            depresjon-dataset/data/control/control_1.csv ...
#            depresjon-dataset/data/scores.csv
# scores.csv: number=condition_1/condition_2/control_1 etc.
# ================================================================
import os, glob
import pandas as pd
import numpy as np

def load_depresjon(base_dir):
    scores_path = os.path.join(base_dir, "scores.csv")
    scores      = pd.read_csv(scores_path)
    scores.columns = [c.strip().lower() for c in scores.columns]
    print("scores.csv columns:", list(scores.columns))
    print(scores.head(3))

    # Build lookup: pid_string -> (madrs1, madrs2)
    madrs_lookup = {}
    for _, row in scores.iterrows():
        key = str(row["number"]).strip()
        m1  = float(row["madrs1"]) if str(row.get("madrs1","NA")).strip() not in ["NA","nan",""] else np.nan
        m2  = float(row["madrs2"]) if str(row.get("madrs2","NA")).strip() not in ["NA","nan",""] else np.nan
        madrs_lookup[key] = (m1, m2)

    participants = []

    for group in ["condition", "control"]:
        folder = os.path.join(base_dir, group)
        csvs   = sorted(glob.glob(os.path.join(folder, "*.csv")))
        print(f"\n{group}: {len(csvs)} files found")

        for csv_path in csvs:
            pid = os.path.splitext(os.path.basename(csv_path))[0]
            try:
                df = pd.read_csv(csv_path)
                df.columns = [c.strip().lower() for c in df.columns]

                if "timestamp" not in df.columns or "activity" not in df.columns:
                    print(f"  SKIP {pid}: columns={list(df.columns)}")
                    continue

                madrs1, madrs2 = madrs_lookup.get(pid, (np.nan, np.nan))
                madrs_mean     = np.nanmean([madrs1, madrs2])

                daily = ndi.extract_daily_features(df)

                if len(daily) < 5:
                    print(f"  SKIP {pid}: only {len(daily)} days")
                    continue

                participants.append(dict(
                    pid          = pid,
                    group        = group,
                    is_depressed = int(group == "condition"),
                    madrs1       = madrs1,
                    madrs2       = madrs2,
                    madrs_mean   = madrs_mean,
                    n_days       = len(daily),
                    days_df      = daily,
                ))
                print(f"  OK {pid}: {len(daily)} days  MADRS={madrs_mean:.0f}" if not np.isnan(madrs_mean)
                      else f"  OK {pid}: {len(daily)} days  MADRS=N/A")

            except Exception as e:
                print(f"  ERROR {pid}: {e}")

    return participants

depresjon_participants = load_depresjon(DEPRESJON_DIR)

condition_p = [p for p in depresjon_participants if p["group"] == "condition"]
control_p   = [p for p in depresjon_participants if p["group"] == "control"]
print("  NOTE: madrs_mean = NaN for control participants — expected, DEPRESJON administered MADRS to condition group only.")
print(f"\nLoaded total: {len(depresjon_participants)} participants")
print(f"  Condition (depressed): {len(condition_p)}")
print(f"  Control   (healthy)  : {len(control_p)}")
if condition_p:
    print(f"  Median days condition: {np.median([p['n_days'] for p in condition_p]):.0f}")
    valid_madrs = [p['madrs_mean'] for p in condition_p if not np.isnan(p['madrs_mean'])]
    if valid_madrs:
        print(f"  MADRS range: {min(valid_madrs):.0f} to {max(valid_madrs):.0f}")
if control_p:
    print(f"  Median days control  : {np.median([p['n_days'] for p in control_p]):.0f}")

In [ ]:
# ================================================================
# CELL 4  BUILD DEPRESJON FEATURE MATRIX
# FIX 2: neutro_df built here is FOR VISUALISATION ONLY.
# All CV metrics rebuild features per-fold in Cell 8.
# FIX 3: calls ndi.build_neutrosophic_features (module is
# authoritative). Inline decomposition deleted.
# ================================================================

FEAT_NAMES = ndi.DEPRESJON_FEATURE_NAMES
DOMAIN_MAP = ndi.DEPRESJON_DOMAIN_MAP

def build_participant_neutrosophic(p):
    """
    For one participant: extract feature matrix, apply neutrosophic
    decomposition using full-series train_mask (visualisation only).
    WARNING: Do NOT use neutro_df from this function in any CV result.
    All CV evaluation rebuilds features inside each fold (Cell 8).
    """
    df = p["days_df"].copy().reset_index(drop=True)
    for col in FEAT_NAMES:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0

    X = df[FEAT_NAMES].values
    n = len(X)

    # Full-series train_mask: all True — avoids the ValueError in
    # build_neutrosophic_features while making clear this is in-sample.
    # VISUALISATION ONLY — do not use for any headline metric.
    train_mask_full = np.ones(n, dtype=bool)
    neutro_df = ndi.build_neutrosophic_features(
        X, FEAT_NAMES, train_mask=train_mask_full)

    result = pd.concat([df[["date"]].reset_index(drop=True),
                        neutro_df.reset_index(drop=True)], axis=1)
    result["pid"]          = p["pid"]
    result["group"]        = p["group"]
    result["is_depressed"] = p["is_depressed"]
    result["madrs_mean"]   = p["madrs_mean"]
    for col in FEAT_NAMES:
        result[col] = df[col].values
    return result

print("Computing neutrosophic features (visualisation only)...")
neutro_frames = []
for p in depresjon_participants:
    try:
        nf = build_participant_neutrosophic(p)
        neutro_frames.append(nf)
        p["neutro_df"] = nf
    except Exception as e:
        print(f"  ERROR {p['pid']}: {e}")

depresjon_all = pd.concat(neutro_frames, ignore_index=True)
print(f"Total rows: {len(depresjon_all)}")
print("NOTE: neutro_df is for visualisation only. CV metrics use per-fold features.")

In [ ]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET   [v1.4: two changes, marked 'v1.4' below]
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # v1.4: wellness is NOT merged into days_df (v1.4 memo, Section 4). It is built as a separate
    # table under rules W1-W4 in cell W-1 and joined only when the Task 3 target is built.

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = pd.to_numeric(row.iloc[0][hr_col], errors="coerce")   # v1.4: p12's cell is ' ' (v1.4 memo, Sec. 2); not a model feature

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")

In [ ]:
# ================================================================
# CELL 6  PMData FEATURE ENGINEERING + WELLNESS MERGE   [v1.4: wellness merge removed, marked 'v1.4']
# Defines feature set, builds neutrosophic features, and crucially
# merges wellness scores as the external target column.
# ================================================================
import numpy as np
import pandas as pd

if not pmdata_participants:
    print("No PMData participants loaded. Check PMDATA_DIR.")
    pmdata_all = pd.DataFrame()
    PMDATA_FEATURE_NAMES = []
    PMDATA_DOMAIN_MAP    = {}
    PMDATA_WELLNESS      = []
else:
    # Collect all columns across participants
    sample_cols = set()
    for p in pmdata_participants:
        sample_cols.update(p["days_df"].columns.tolist())
    sample_cols.discard("date")

    PMDATA_PHYS = [c for c in [
        "resting_hr","steps","active_minutes_total",
        "lightly_active_min","moderately_active_min",
        "very_active_min","sedentary_min"
    ] if c in sample_cols]

    PMDATA_SLEEP = [c for c in [
        "sleep_efficiency","sleep_minutes","sleep_score"
    ] if c in sample_cols]

    # Wellness columns — these become the EXTERNAL TARGET for lambda search
    PMDATA_WELLNESS = [c for c in [
        "fatigue","mood","readiness","sleep_quality","stress","soreness"
    ] if c in sample_cols]

    PMDATA_FEATURE_NAMES = PMDATA_PHYS + PMDATA_SLEEP
    PMDATA_DOMAIN_MAP    = {"physiology": PMDATA_PHYS, "sleep": PMDATA_SLEEP}

    print("PMData physiology features:", PMDATA_PHYS)
    print("PMData sleep features:     ", PMDATA_SLEEP)
    print("PMData wellness/target cols:", PMDATA_WELLNESS)
    print("Total NDI features:        ", len(PMDATA_FEATURE_NAMES))

    # v1.4: the wellness re-load and merge that stood here (UTC date, one row per REPORT) is removed: it is what
    # duplicated 18 days in v1-v1.3 (v1.4 memo). days_df stays one row per Fitbit date; wellness is built in W-1.
    print("v1.4: wellness kept out of days_df (separate table, cell W-1)")

    # Build neutrosophic features for each participant
    print("\nComputing neutrosophic features for PMData participants...")
    pmdata_frames = []
    for p in pmdata_participants:
        try:
            df = p["days_df"].copy().reset_index(drop=True)

            for col in PMDATA_FEATURE_NAMES:
                if col not in df.columns:
                    df[col] = np.nan
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].fillna(df[col].median())

            X        = df[PMDATA_FEATURE_NAMES].values
            train_mask_full = np.ones(len(X), dtype=bool)
            neutro   = ndi.build_neutrosophic_features(X, PMDATA_FEATURE_NAMES,
                                            train_mask=train_mask_full)
            result   = pd.concat([df[["date"]].reset_index(drop=True),
                                   neutro.reset_index(drop=True)], axis=1)
            result["pid"]    = p["pid"]
            result["max_hr"] = p["max_hr"]

            # Keep raw features AND wellness columns
            for col in PMDATA_FEATURE_NAMES + PMDATA_WELLNESS:
                if col in df.columns:
                    result[col] = df[col].values

            pmdata_frames.append(result)
            p["neutro_df"] = result

            well_coverage = {c: df[c].notna().sum() for c in PMDATA_WELLNESS
                             if c in df.columns}
            print(f"  {p['pid']}: {len(result)} days  "
                  f"NDI={result['NDI'].mean():.4f}  "
                  f"wellness={well_coverage}")

        except Exception as e:
            print(f"  ERROR {p['pid']}: {e}")

    if pmdata_frames:
        pmdata_all = pd.concat(pmdata_frames, ignore_index=True)
        print(f"\nPMData total rows: {len(pmdata_all)}")
        print(f"NDI range: [{pmdata_all['NDI'].min():.4f},"
              f"{pmdata_all['NDI'].max():.4f}]")
        for wcol in PMDATA_WELLNESS:
            if wcol in pmdata_all.columns:
                vals = pmdata_all[wcol].dropna()
                if len(vals) > 0:
                    print(f"  {wcol}: mean={vals.mean():.2f}  "
                          f"range=[{vals.min():.0f},{vals.max():.0f}]  "
                          f"n={len(vals)}")
    else:
        print("No PMData frames built.")
        pmdata_all           = pd.DataFrame()
        PMDATA_FEATURE_NAMES = []
        PMDATA_DOMAIN_MAP    = {}
        PMDATA_WELLNESS      = []

In [ ]:
# D-2 (v1.4) One row per local calendar date: every days_df must have unique, consecutive dates (v1.4 memo, Section 4)
# days_df comes straight from the Fitbit daily files (CELL 5); wellness is not merged into it any more (CELL 5, CELL 6).
# Any duplicated, unparseable or missing date stops the run here, with the offending dates printed.
import numpy as np, pandas as pd
_WELL_COLS = {"fatigue", "mood", "readiness", "sleep_quality", "stress", "soreness", "effective_time_frame"}
_rows, _bad = [], []
for p in pmdata_participants:
    d = pd.to_datetime(p["days_df"]["date"], errors="coerce")
    dd = d.dropna()
    steps = dd.diff().dt.days.dropna()
    gaps = [(str(a.date()), str(b.date())) for a, b, s in zip(dd.iloc[:-1], dd.iloc[1:], steps) if s != 1]
    leaked = sorted(_WELL_COLS & set(p["days_df"].columns))
    _rows.append(dict(pid=p["pid"], rows=len(d), first_date=str(dd.min().date()), last_date=str(dd.max().date()),
                      unparseable_dates=int(d.isna().sum()), duplicated_dates=int(d.duplicated().sum()),
                      non_consecutive_steps=len(gaps), wellness_columns_in_days_df=len(leaked)))
    if d.isna().any() or d.duplicated().any() or gaps or leaked:
        _bad.append((p["pid"], int(d.isna().sum()), int(d.duplicated().sum()), gaps[:10], leaked))
DATE_CHECK = pd.DataFrame(_rows)
print(DATE_CHECK.to_string(index=False))
if _bad:
    for pid, n_nat, n_dup, g, lk in _bad:
        print(f"  {pid}: unparseable {n_nat}, duplicated {n_dup}, steps != 1 day (previous, next): {g}, wellness columns: {lk}")
    raise RuntimeError("STOP: days_df is not one row per consecutive calendar date (v1.4 memo, Section 4). "
                       "Send the lines above to ma'am; do not fix by hand.")
print(f"D-2 PASSED: {len(DATE_CHECK)} participants, {int(DATE_CHECK['rows'].sum())} rows, "
      "every date unique and consecutive, no wellness column in days_df")

In [ ]:
# W-1 (v1.4) Wellness table under W1-W4, kept OUT of days_df (v1.4 memo, Sections 3-4)
# One row per participant and local date; joined only when the Task 3 target is built. Tasks 1-2 do not use it.
# Same functions as W-0 in NDI_v1_4_prechecks.ipynb. Built here, early, so a problem stops the run in minutes;
# the files are written by X-1 into results_v1.4.
import os
import numpy as np, pandas as pd

# ---- Wellness rules W1-W4 (v1.4 memo, Section 3). Identical in NDI_v1_4_prechecks.ipynb (W-0) and NDI_v1_4.ipynb (W-1).
WELLNESS_TZ = "Europe/Oslo"
W_ITEMS_1_5 = ["mood", "stress", "fatigue", "sleep_quality", "soreness"]   # scored 1-5: 0 = missing (W2)
W_ITEMS_ALL = W_ITEMS_1_5 + ["readiness"]                                  # readiness runs 0-10: 0 is valid
W_RANGE = {**{c: (1, 5) for c in W_ITEMS_1_5}, "readiness": (0, 10)}

def load_wellness_reports(pmdata_dir, pid):
    """Every report in pmsys/wellness.csv, untouched, with its UTC and local (W1) calendar date."""
    path = os.path.join(pmdata_dir, pid, "pmsys", "wellness.csv")
    if not os.path.isfile(path):
        return None
    w = pd.read_csv(path)
    w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
    raw = w["effective_time_frame"].astype(str).str.strip()
    ts = pd.to_datetime(raw, errors="coerce", utc=True)
    rep = pd.DataFrame({"pid": pid, "timestamp": raw.values, "ts_utc": ts.values})
    rep["ts_utc"] = pd.to_datetime(rep["ts_utc"], utc=True)
    rep["tz_aware"] = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True).values
    rep["utc_date"] = rep["ts_utc"].dt.strftime("%Y-%m-%d")
    rep["local_date"] = rep["ts_utc"].dt.tz_convert(WELLNESS_TZ).dt.strftime("%Y-%m-%d")      # W1
    for c in W_ITEMS_ALL:
        rep[c] = pd.to_numeric(w[c], errors="coerce").values if c in w.columns else np.nan
    return rep

def apply_wellness_rules(rep):
    """W1-W4 for one participant. Returns (daily table: one row per local date, log dict)."""
    pid = rep["pid"].iloc[0] if len(rep) else None
    bad_ts = rep["ts_utc"].isna() | ~rep["tz_aware"]
    r = rep[~bad_ts].copy()
    log = dict(pid=pid, reports=len(rep), timestamps_unparseable_or_without_timezone=int(bad_ts.sum()),
               reports_moved_by_W1=int((r["utc_date"] != r["local_date"]).sum()))
    for c in W_ITEMS_ALL:                                   # anything other than 0 outside the scale
        lo, hi = W_RANGE[c]
        v = r[c]
        log[f"out_of_range_{c}"] = int((v.notna() & (v != 0) & ((v < lo) | (v > hi) | (v != v.round()))).sum())
    for c in W_ITEMS_1_5:
        log[f"zeros_{c}"] = int((r[c] == 0).sum())
    blank = (r[W_ITEMS_1_5] == 0).all(axis=1)              # W3: all five 1-5 items are 0 -> blank submission
    log["reports_dropped_W3"] = int(blank.sum())
    r = r[~blank].copy()
    for c in W_ITEMS_1_5:                                   # W2: 0 = missing on the 1-5 items
        log[f"zeros_set_missing_W2_{c}"] = int((r[c] == 0).sum())
        r.loc[r[c] == 0, c] = np.nan
    r = r.sort_values("ts_utc", kind="mergesort")
    g = r.groupby("local_date", sort=True)
    daily = g[W_ITEMS_ALL].last()                           # W4: last valid value of the day, item by item
    daily.insert(0, "reports_that_day", g.size())
    daily = daily.reset_index().rename(columns={"local_date": "date"})
    daily.insert(0, "pid", pid)
    log["local_days_with_a_report"] = len(daily)
    log["days_with_2plus_reports_W4"] = int((daily["reports_that_day"] >= 2).sum())
    for c in W_ITEMS_ALL:
        nvalid = g[c].count()
        nuniq = g[c].nunique()
        log[f"W4_days_choosing_{c}"] = int((nvalid >= 2).sum())         # days where W4 picked among valid values
        log[f"W4_days_values_differ_{c}"] = int((nuniq >= 2).sum())     # ... and the values disagreed
    return daily, log

def build_wellness_tables(pmdata_dir, pids):
    tables, logs = {}, []
    for pid in pids:
        rep = load_wellness_reports(pmdata_dir, pid)
        if rep is None or rep.empty:
            logs.append(dict(pid=pid, reports=0))
            continue
        tables[pid], lg = apply_wellness_rules(rep)
        logs.append(lg)
    log = pd.DataFrame(logs).fillna(0)
    for c in log.columns:
        if c != "pid":
            log[c] = log[c].astype(int)
    return tables, log

def wellness_zero_table(log):
    """Deliverable 2: zeros per participant and 1-5 item (all reports), blank reports W3 drops, zero share."""
    z = log[["pid", "reports"] + [f"zeros_{c}" for c in W_ITEMS_1_5] + ["reports_dropped_W3"]].copy()
    z["zeros_total"] = z[[f"zeros_{c}" for c in W_ITEMS_1_5]].sum(axis=1)
    z["zero_share_of_item_values"] = (z["zeros_total"] / (5 * z["reports"]).where(z["reports"] > 0)).round(3)
    z["zeros_left_after_W3"] = z["zeros_total"] - 5 * z["reports_dropped_W3"]
    return z

_pids = sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p"))
wellness_tables, wellness_log = build_wellness_tables(PMDATA_DIR, _pids)
wellness_daily = pd.concat(list(wellness_tables.values()), ignore_index=True) if wellness_tables else pd.DataFrame()
wellness_zeros = wellness_zero_table(wellness_log)
_chk = ["timestamps_unparseable_or_without_timezone"] + [f"out_of_range_{c}" for c in W_ITEMS_ALL]
_bad = wellness_log[wellness_log[_chk].sum(axis=1) > 0]
if not _bad.empty:
    print(_bad[["pid"] + _chk].to_string(index=False))
    raise RuntimeError("STOP: wellness timestamps or values the W1-W4 rules do not cover (see above). Report to ma'am.")
assert not wellness_daily.duplicated(["pid", "date"]).any(), "wellness table is not one row per local date"
_fit_dates = {p["pid"]: set(pd.to_datetime(p["days_df"]["date"]).dt.strftime("%Y-%m-%d")) for p in pmdata_participants}
wellness_log["local_days_outside_fitbit_dates"] = [
    int((~wellness_daily.loc[wellness_daily["pid"] == pid, "date"].isin(_fit_dates[pid])).sum()) if pid in _fit_dates else np.nan
    for pid in wellness_log["pid"]]
print(f"W-1: {len(wellness_daily)} participant-days from {int(wellness_log['reports'].sum())} reports | "
      f"W1 moved {int(wellness_log['reports_moved_by_W1'].sum())} reports | W3 dropped {int(wellness_log['reports_dropped_W3'].sum())} | "
      f"W2 set {int(wellness_zeros['zeros_left_after_W3'].sum())} zeros to missing | "
      f"days with 2+ reports (W4): {int(wellness_log['days_with_2plus_reports_W4'].sum())}, "
      f"mood values differing on {int(wellness_log['W4_days_values_differ_mood'].sum())} of them")

In [ ]:
# ================================================================
# CELL 6.5  F_SCALE PRE-SPECIFICATION SWEEP  (follow-up note, "The fix")
# Run this BEFORE Cell 7 and BEFORE inspecting any CV/rho result.
#
# Pre-specified rule (fixed in advance; do not revise after seeing
# the numbers): sweep SCALE over {1, 2, 3, 4, 5}. For each candidate,
# rebuild F for every feature and participant in both datasets, pool
# all rows, and compute per dataset:
#   F_IQR       = interquartile range of pooled F  (higher = less
#                 saturated / more graded)
#   |Sp(T,F)|   = |Spearman(T, F)| on pooled T/F
# Choose the SCALE that MAXIMISES the mean F_IQR across datasets,
# subject to the WORST-CASE (max across datasets) |Sp(T,F)| staying
# < 0.7. If no candidate satisfies the constraint on both datasets,
# keep the current CONFIG["F_SCALE"] and flag for manual review
# rather than silently picking a value that fails the independence
# check on one dataset.
# ================================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SCALE_GRID = [1, 2, 3, 4, 5]
_original_f_scale = ndi.CONFIG["F_SCALE"]
sweep_rows = []

for scale in SCALE_GRID:
    ndi.CONFIG["F_SCALE"] = scale   # neutrosophic_decompose reads CONFIG at call time

    for dataset_name, participants, feat_names in [
        ("DEPRESJON", depresjon_participants, FEAT_NAMES),
        ("PMData",    pmdata_participants,    PMDATA_FEATURE_NAMES),
    ]:
        if not participants:
            continue
        T_pool, F_pool = [], []
        for p in participants:
            df = p["days_df"].copy().reset_index(drop=True)
            for col in feat_names:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = 0.0
            X = df[feat_names].values
            n = len(X)
            train_mask_full = np.ones(n, dtype=bool)  # same in-sample mode as Cell 4/6
            neutro_df = ndi.build_neutrosophic_features(
                X, feat_names, train_mask=train_mask_full)
            T_pool.append(neutro_df["Depression_T"].values)
            F_pool.append(neutro_df["Depression_F"].values)

        T_all = np.concatenate(T_pool)
        F_all = np.concatenate(F_pool)
        valid = ~(np.isnan(T_all) | np.isnan(F_all))
        T_all, F_all = T_all[valid], F_all[valid]

        f_iqr = float(np.percentile(F_all, 75) - np.percentile(F_all, 25))
        sp, _ = spearmanr(T_all, F_all)

        sweep_rows.append(dict(scale=scale, dataset=dataset_name,
                                F_IQR=f_iqr, abs_Sp_TF=abs(sp)))

sweep_df = pd.DataFrame(sweep_rows)
print("=" * 65)
print("F_SCALE PRE-SPECIFICATION SWEEP")
print("=" * 65)
print(sweep_df.to_string(index=False))

agg = (sweep_df.groupby("scale")
       .agg(mean_F_IQR=("F_IQR", "mean"),
            max_abs_Sp_TF=("abs_Sp_TF", "max"))
       .reset_index())
agg["passes_constraint"] = agg["max_abs_Sp_TF"] < 0.7
print()
print(agg.to_string(index=False))

qualifying = agg[agg["passes_constraint"]]
if len(qualifying) > 0:
    best_row = qualifying.loc[qualifying["mean_F_IQR"].idxmax()]
    CHOSEN_SCALE = float(best_row["scale"])
    print(f"\nCHOSEN F_SCALE = {CHOSEN_SCALE}  "
          f"(mean F_IQR={best_row['mean_F_IQR']:.3f}, "
          f"worst-case |Sp(T,F)|={best_row['max_abs_Sp_TF']:.3f} < 0.7)")
else:
    CHOSEN_SCALE = _original_f_scale
    print(f"\nNo SCALE in {SCALE_GRID} keeps |Sp(T,F)| < 0.7 on BOTH "
          f"datasets -- keeping current CONFIG['F_SCALE']="
          f"{_original_f_scale} and flagging for manual review "
          f"(widen the grid, or revisit the normaliser).")

# FREEZE. This is the only place F_SCALE should be set after this
# point. Chosen by the rule above, before any CV/SHAP/lambda result
# was inspected -- do not hand-tune this later after seeing rho.
ndi.CONFIG["F_SCALE"] = CHOSEN_SCALE
print(f"\nndi.CONFIG['F_SCALE'] frozen at {CHOSEN_SCALE}.")
if CHOSEN_SCALE != _original_f_scale:
    print("This differs from the value already baked into "
          "depresjon_all / pmdata_all (built in Cell 4 / Cell 6) -- "
          "RE-RUN those two cells now so Cell 7's verification "
          "reflects the frozen value. Per-fold CV cells (Component 2 "
          "onward) rebuild features from raw data on every call, so "
          "they pick up CONFIG['F_SCALE'] automatically.")
else:
    print("Matches the value already used in Cell 4 / Cell 6 -- no "
          "rebuild needed.")

In [ ]:
# ================================================================
# CELL 14.4  V2' — POPULATION VOLATILITY NORM + FOLD-WISE FRAME
# (V2' spec Sec 4) Population statistic is now the SPREAD OF PER-
# PERSON VOLATILITIES across donors, not raw values. Leakage rule
# unchanged: p never enters its own donor pool.
# ================================================================
def _compute_population_volatility_norm(participants, held_out_pid, feat_names, has_labels):
    donors = [q for q in participants if q["pid"] != held_out_pid]
    if has_labels:
        donors = [q for q in donors if q.get("group") == "control"]
    donor_ids = [q["pid"] for q in donors]

    assert held_out_pid not in donor_ids, \
        f"LEAKAGE: {held_out_pid} is in its own donor pool"

    donor_vols = []
    for q in donors:
        dfq = q["days_df"]
        vols = []
        for col in feat_names:
            v = pd.to_numeric(dfq[col], errors="coerce") if col in dfq.columns \
                else pd.Series(np.nan, index=dfq.index)
            vols.append(v.std())
        donor_vols.append(vols)
    donor_vols = np.array(donor_vols, dtype=float)  # (n_donors, d)

    pop_vol_median = np.nanmedian(donor_vols, axis=0)
    pop_vol_mad    = np.nanmedian(np.abs(donor_vols - pop_vol_median), axis=0) + 1e-8
    return pop_vol_median, pop_vol_mad, donor_ids


def build_v2prime_diagnostic_frame_foldwise(participants, feat_names, has_labels):
    """Fold-wise (not whole-series) -- V2's degenerate-frame lesson
    applies here too: F must vary within a participant to test
    independence against."""
    frames = []
    for p in participants:
        df_raw = p.get("days_df")
        if df_raw is None:
            continue
        df_raw = df_raw.copy().reset_index(drop=True)
        for col in feat_names:
            if col in df_raw.columns:
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
                df_raw[col] = df_raw[col].fillna(df_raw[col].median())
            else:
                df_raw[col] = 0.0
        n = len(df_raw)
        if n < 15:
            continue
        X_raw = df_raw[feat_names].values
        t0 = min(10, max(5, n // 3))

        pop_vol_median, pop_vol_mad, _ = _compute_population_volatility_norm(
            participants, p["pid"], feat_names, has_labels)

        try:
            fold_info = ndi.blocked_tscv(
                X_raw, np.zeros(n), ndi.rf_factory, T0=t0, return_folds=True)
            folds = fold_info.get("folds", [])
        except Exception:
            continue

        rows = []
        for fold in folds:
            tr_idx, te_idx = fold
            tr_idx = np.array(tr_idx); te_idx = np.array(te_idx)
            if int(te_idx.min()) - int(tr_idx.max()) < 6:
                continue
            train_mask = np.zeros(n, dtype=bool)
            train_mask[tr_idx] = True

            neutro_fold = ndi.build_neutrosophic_features(
                X_raw, feat_names, train_mask=train_mask,
                f_mode="population_vol",
                pop_vol_median=pop_vol_median, pop_vol_mad=pop_vol_mad)
            rows.append(neutro_fold.iloc[te_idx])

        if not rows:
            continue
        pf = pd.concat(rows, ignore_index=True)
        pf["pid"]   = p["pid"]
        pf["group"] = p.get("group", "pmdata")
        frames.append(pf)

    return pd.concat(frames, ignore_index=True)

In [ ]:
# ================================================================
# CELL 14.10  ITEM 2a -- V3_BEST_LAMBDA (frozen; run completed fully
# before this session disconnected -- both datasets printed, no
# need to rerun the 12hr search)
#
#   PMData:    best_lambda=(0.20, 0.15, 0.65)  mean_rho=0.3834
#              original_mean_rho=0.3198  n_stable=13  Robust
#   DEPRESJON: best_lambda=(0.00, 1.00, 0.00)  mean_rho=0.6325
#              original_mean_rho=0.2962  n_stable=1
#              NOT robust -- corner solution, near-zero stability
#              radius. Not used downstream (V3_BEST_LAMBDA freezes
#              PMData's), kept here only for the write-up's caveat.
# ================================================================
V3_BEST_LAMBDA = (0.20, 0.15, 0.65)

pmdata_lambda_result = dict(
    best_lambda=V3_BEST_LAMBDA, best_mean_rho=0.3834,
    original_mean_rho=0.3198, n_stable=13,
    verdict="Robust — weights stable across neighbourhood")

depresjon_lambda_result = dict(
    best_lambda=(0.0, 1.0, 0.0), best_mean_rho=0.6325,
    original_mean_rho=0.2962, n_stable=1,
    verdict="Per-subject tuning recommended (small stability radius) — corner solution, not robust")

print(f"V3_BEST_LAMBDA (frozen from PMData primary calibration): {V3_BEST_LAMBDA}")

## Repository, Task 3 config and provenance

In [ ]:
# G-0 (Task 3) Run from inside the repository so T3-0 can see git; same as in NDI_v1_4.ipynb except RUN_TAG
# Works with the private repository: if a plain clone/fetch fails it asks once for a read-only token
# (typed into a hidden prompt, never stored in the notebook or in .git/config), then checks out the
# tag so the run's HEAD is the tagged commit. Idempotent.
import os, subprocess, getpass
REPO_URL = "https://github.com/hpatel09112004/NDI_Depression_Risk_Index.git"
REPO_DIR = "/content/NDI_Depression_Risk_Index"
RUN_TAG = "v1.4-task3"
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

def _git_run(args, secret=None):
    r = subprocess.run(args, capture_output=True, text=True, env=_env)
    msg = (r.stderr or "")[-600:]
    return r.returncode, (msg.replace(secret, "***") if secret else msg)

def _token_url():
    tok = getpass.getpass("Private repository: paste the read-only GitHub token (hidden, not saved): ").strip()
    return REPO_URL.replace("https://", f"https://hpatel09112004:{tok}@"), tok

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _rc, _msg = _git_run(["git", "clone", "--quiet", REPO_URL, REPO_DIR])
    if _rc != 0:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "clone", "--quiet", _url, REPO_DIR], secret=_tok)
        assert _rc == 0, "git clone failed (token or URL?):\n" + _msg
        _git_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
else:
    _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet"])
    _have = _git_run(["git", "-C", REPO_DIR, "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0
    if _rc != 0 and not _have:
        _url, _tok = _token_url()
        _rc, _msg = _git_run(["git", "-C", REPO_DIR, "fetch", "--tags", "--quiet", _url], secret=_tok)
        assert _rc == 0, "git fetch failed (token or URL?):\n" + _msg
os.chdir(REPO_DIR)
if _git_run(["git", "rev-parse", "-q", "--verify", f"refs/tags/{RUN_TAG}"])[0] == 0:
    _rc, _msg = _git_run(["git", "checkout", "--quiet", "--force", RUN_TAG])   # --force: a config file left by an untagged attempt is replaced by the tagged one
    assert _rc == 0, f"could not check out {RUN_TAG}:\n" + _msg
    print(f"Checked out tag {RUN_TAG} (detached HEAD at the tagged commit)")
else:
    print(f"Tag {RUN_TAG} not found yet: T3-0 will write the config and stop.")
print("Working directory:", os.getcwd())
print("HEAD:", subprocess.run(["git", "log", "-1", "--format=%H %cI %s"], capture_output=True, text=True).stdout.strip())
print("Tags:", subprocess.run(["git", "tag", "--list"], capture_output=True, text=True).stdout.split())

In [ ]:
# T3-0 Task 3 config, provenance (tag v1.4-task3). Analysis settings are inherited unchanged from v1.4.
# Task 3 (revision analyses memo): independent self-report target from PMData wellness.csv.
# Primary next-day mood, secondary next-day stress, under wellness rules W1-W4 (v1.4 memo, Section 3).
# v1.4 review memo: runs only after the sleep-score check passed (Section 3); persistence references (Section 4,
# items 3-4) are in the config, so they are fixed before any Task 3 result.
import os, json, time, subprocess, datetime, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon, bootstrap as scipy_bootstrap
from sklearn.linear_model import Ridge

_REQUIRED = ["ndi", "pmdata_participants", "PMDATA_FEATURE_NAMES", "V3_BEST_LAMBDA", "RANDOM_SEED", "RESULTS_DIR",
             "PMDATA_DIR", "_compute_population_volatility_norm", "DATE_CHECK", "wellness_daily", "wellness_log"]
_missing = [g for g in _REQUIRED if g not in globals()]
assert not _missing, f"Run the notebook from the top first. Missing: {_missing}"

def _git(*args):
    try:
        return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None

_BASE_DIR = os.path.dirname(RESULTS_DIR)
RESULTS_V1_4_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.4")      # read-only
RESULTS_T3_DIR = os.path.join(_BASE_DIR, "results_v1.4_task3")
_need14 = ["config_v1_4.json", "table1_corrected_self_target_per_participant.csv", "raw_target_corrected_per_participant.csv",
           "task1_table2_bounded_null.csv", "task1_absolute_rho_both_targets.csv", "wellness_daily_W1_W4.csv",
           "supplementary_without_p12_self_target_per_participant.csv",
           "supplementary_without_p12_raw_target_per_participant.csv", "task2_one_at_a_time.csv",
           "task2_waterfall_cumulative.csv", "task2_table4_inflation_column.csv"]
_absent = [f for f in _need14 if not os.path.isfile(os.path.join(RESULTS_V1_4_ARCHIVE, f))]
assert not _absent, f"results_v1.4 files needed by Task 3 are missing: {_absent}"
RESULTS_V1_2_ARCHIVE = os.path.join(_BASE_DIR, "results_v1.2")      # read-only (Table 4: L2 under the trailing target)
assert os.path.isfile(os.path.join(RESULTS_V1_2_ARCHIVE, "task2_table4_inflation_column.csv")), "results_v1.2 Table 4 file missing"

# v1.4 analysis settings: the archived v1.4 run's config must equal the config under tag v1.4
V1_2_CONFIG = json.load(open(os.path.join(RESULTS_V1_4_ARCHIVE, "config_v1_4.json")))   # name kept for V12-1/V12-2
_cfg14_tag = _git("show", "v1.4:config_v1_4.json")
assert _cfg14_tag is not None and json.loads(_cfg14_tag) == V1_2_CONFIG, \
    "results_v1.4/config_v1_4.json differs from the config under tag v1.4 -- STOP"
assert float(ndi.CONFIG["F_SCALE"]) == V1_2_CONFIG["F_SCALE_FROZEN"], "F_SCALE is not the frozen 1.0 -- STOP"
_ssv_path = os.path.join(_BASE_DIR, "results_diagnostics", "sleep_score_check_verdict.json")
assert os.path.isfile(_ssv_path), "Run NDI_sleep_score_check.ipynb first (v1.4 review memo, Section 3)"
SLEEP_SCORE_CHECK = json.load(open(_ssv_path))
assert SLEEP_SCORE_CHECK["verdict"] == "PASS", f"sleep-score check did not pass: {SLEEP_SCORE_CHECK['reason']} -- STOP"

TASK3_CONFIG = dict(
    VERSION="v1.4-task3",
    BASE_VERSION="v1.4", BASE_TAG="v1.4",
    ANALYSIS_SETTINGS="inherited unchanged from config_v1_4.json under tag v1.4: features, fold-local statistics, "
                      "fold generator (T0 rule, embargo g = 6, fold size 1), purge rule, four arms, models, seed, "
                      "bootstrap, SESOI. Only the target column changes.",
    TARGETS=dict(primary="mood", secondary="stress"),
    TARGETS_EXCLUDED="sleep quality and sleep duration (share structure with the Fitbit sleep features); readiness, "
                     "fatigue and soreness (sport-load constructs) -- revision analyses memo, Task 3",
    TARGET_SOURCE="pmsys/wellness.csv through the W1-W4 table built in cell W-1 (config_v1_4.json WELLNESS_RULES); "
                  "no wellness.csv field enters any arm's features (asserted in T3-1)",
    TARGET_FORM="raw next-day value, not smoothed: row s (Fitbit calendar day d_s) is scored against the W1-W4 value "
                "of local day d_s + 1",
    DAY_ALIGNMENT="report dated by its local Europe/Oslo calendar day (W1), the day boundary of the Fitbit daily files",
    TARGET_ALIGNMENT="next_day",     # target support of row s = {s + 1}
    PURGE="v1.4 purge rule applied with target support {s + 1}; with embargo g = 6 it removes no training row",
    EMBARGO_G=6,
    MISSING_TARGET="test day without a next-day value: dropped from scoring, not imputed; training rows without a "
                   "target: dropped from that fold's fit",
    ELIGIBILITY=dict(min_scored_test_folds=10, min_distinct_target_values=3,
                     applied_to="each target separately, on that target's values over the scored test days "
                                "(the memo states the rule for mood; stress uses the same rule on stress values)"),
    MIN_ELIGIBLE_FOR_TESTS=10,       # fewer eligible participants -> descriptive only, no P, q or TOST
    STATISTICS="as Task 1 (V12-6 functions, unchanged): mean delta rho with BCa 95% CI (10,000 resamples, seed 42); "
               "Hodges-Lehmann median with BCa 95% CI; paired Wilcoxon P; BH q across the four neutrosophic-vs-"
               "baseline comparisons within each target; TOST at +/-0.10 on the 90% CI; advantage >= 0.10 rejected "
               "if the upper 90% bound < 0.10; absolute rho per arm with BCa 95% CI and one-sample Wilcoxon vs 0. "
               "V3 vs V-signed reported without q.",
    ARTIFACT_BOUND="upper 95% bound of each Task 3 comparison vs the self-derived advantage computed on the SAME "
                   "eligible participants (mean over them, from results_v1.4/table1_corrected_self_target_per_"
                   "participant.csv; without-p12 rows: from results_v1.4/supplementary_without_p12_self_target_per_"
                   "participant.csv). n/a for V3 vs V-signed (no q) -- v1.4 review memo, Section 4 item 1",
    COMPLETE_PAIRS="every eligible participant must have a finite rho in all four arms, so every paired test uses "
                   "the same n; otherwise the run stops (v1.4 review memo, Section 4 item 2)",
    P12="included in the primary analysis (guide's decision, v1.4 review memo Section 1: its exclusion was a loader "
        "bug and it meets the rules fixed before any result). If p12 is eligible for a target, that target is also "
        "reported without p12 (p12 removed from the cohort and from every donor pool) as a sensitivity analysis",
    PERSISTENCE_REFERENCES=dict(
        status="reference rows outside the BH set; fixed here before any Task 3 result (v1.4 review memo, Section 4 "
               "items 3-4)",
        mood_and_stress="forecast for row s = the W1-W4 value of day d_s (today's report) for the next-day value; "
                        "scored on the Task 3 scored test days of the eligible participants where today's value "
                        "exists; per-participant Spearman rho; mean rho with BCa 95% CI; days used; per-participant "
                        "values saved",
        self_derived="the L2 proxy (l2_proxy_rho: forward target, fixed lambda (0.2, 0.3, 0.5), no model) recomputed "
                     "for all v1.4 participants; its mean must equal the v1.4 'Clean + L2 only' value in "
                     "results_v1.4/task2_one_at_a_time.csv; per-participant values saved; paired V3 vs persistence "
                     "(mean delta rho, BCa 95% CI, Wilcoxon P), labelled exploratory",
        raw_feature="today's causal raw composite (the quantity the raw-feature target smooths) as the forecast of "
                    "the raw-feature target; reference only, to show whether that target is predictable at all"),
    L2_UNDER_FORWARD_TARGET="the self-proxy is a naive persistence forecast, not a leak (guide's decision, v1.4 review "
                            "memo Section 2): Figure 4 is redrawn from the L2-removed state; Table 4 shows L2 under "
                            "both targets (v1.2 trailing, v1.4 forward)",
    SLEEP_SCORE_DATING="checked before tagging (results_diagnostics/sleep_score_check_*): v1.4 features kept unchanged",
    REPORTING_FROM_V1_4=["target-swap shift without p12", "Figure 4 redrawn from the L2-removed state",
                         "Table 4 standalone inflation with L2 under both targets",
                         "formatted Table 2 with 'artifact excluded' n/a for V3 vs V-signed",
                         "written to results_v1.4_task3; results_v1.4 stays read-only"],
    W4_LOG="number of scored test days whose target day had 2+ valid values of the item, and on how many they differed",
    FIGURE="Figure 2 as three panels: (A) self-derived and (B) raw-feature targets from results_v1.4, (C) next-day "
           "mood from this run; stress in a supplementary figure",
    CHECKS=["sleep-score check passed (results_diagnostics/sleep_score_check_verdict.json)",
            "v1.4 corrected Table 1 and raw-feature comparison reproduce results_v1.4 exactly, first and last",
            "wellness table equals results_v1.4/wellness_daily_W1_W4.csv",
            "independent recomputation of the mood target from wellness.csv and of V3 / raw-RF on it"],
    REPORTING_DECIMALS=3,
)

TAG = "v1.4-task3"
CONFIG_NAME = "config_task3_v1_4.json"
os.makedirs(RESULTS_T3_DIR, exist_ok=True)
with open(os.path.join(RESULTS_T3_DIR, CONFIG_NAME), "w") as f:
    json.dump(TASK3_CONFIG, f, indent=2)
with open(CONFIG_NAME, "w") as f:    # repo copy -- this is the file that is committed and tagged
    json.dump(TASK3_CONFIG, f, indent=2)

_now = datetime.datetime.now(datetime.timezone.utc)
_tag_commit = _git("rev-list", "-n", "1", TAG)
_tag_time = _git("log", "-1", "--format=%cI", TAG)
_tagged_cfg = _git("show", f"{TAG}:{CONFIG_NAME}")
try:
    _cfg_same = _tagged_cfg is not None and json.loads(_tagged_cfg) == json.loads(json.dumps(TASK3_CONFIG))
except Exception:
    _cfg_same = False
PROVENANCE = dict(version=TASK3_CONFIG["VERSION"], run_started=_now.isoformat(timespec="seconds"),
                  working_directory=os.getcwd(), head_commit=_git("rev-parse", "HEAD"),
                  tag=TAG, tag_commit=_tag_commit, tag_commit_time=_tag_time,
                  config_matches_tagged_config=bool(_cfg_same),
                  tag_time_precedes_run_start=bool(_tag_time and datetime.datetime.fromisoformat(_tag_time) < _now),
                  base_tag_v1_4_commit=_git("rev-list", "-n", "1", "v1.4"),
                  v1_4_config_matches_tag=True, f_scale=float(ndi.CONFIG["F_SCALE"]),
                  sleep_score_check=SLEEP_SCORE_CHECK,
                  pmdata_participants_loaded=[p["pid"] for p in pmdata_participants])
print(json.dumps(PROVENANCE, indent=2))
ALLOW_UNTAGGED_RUN = False           # leave False
if not (bool(_tag_commit) and _cfg_same and PROVENANCE["tag_time_precedes_run_start"]) and not ALLOW_UNTAGGED_RUN:
    raise RuntimeError(
        f"STOP: this run is not tagged. {CONFIG_NAME} has been written to {os.getcwd()}.\n"
        f"Commit it together with this notebook, create tag {TAG} on that commit, then Runtime -> Disconnect and "
        "delete runtime, and Run all. (tag found: %s | config matches tag: %s | tag precedes run: %s)"
        % (bool(_tag_commit), _cfg_same, PROVENANCE["tag_time_precedes_run_start"]))
with open(os.path.join(RESULTS_T3_DIR, "provenance.json"), "w") as f:
    json.dump(PROVENANCE, f, indent=2)
print(f"TAG OK: {TAG} -> {_tag_commit[:10] if _tag_commit else None}, committed {_tag_time}; outputs -> {RESULTS_T3_DIR}")

SEED, N_BOOT, SESOI = V1_2_CONFIG["BOOTSTRAP_SEED"], V1_2_CONFIG["BOOTSTRAP_N"], V1_2_CONFIG["SESOI"]
K, G, MF = V1_2_CONFIG["SMOOTH_K"], V1_2_CONFIG["EMBARGO_G"], V1_2_CONFIG["MIN_FOLDS"]
assert G == TASK3_CONFIG["EMBARGO_G"]
ARMS = ["vsigned", "v3", "rawrf", "persz"]
ARM_LABEL = {"vsigned": "V-signed", "v3": "V3", "rawrf": "Raw-feature RF", "persz": "Personalised Z"}
COMPARISONS = [("v3", "rawrf", "V3 vs raw-RF"), ("v3", "persz", "V3 vs pers-Z"),
               ("vsigned", "rawrf", "V-signed vs raw-RF"), ("vsigned", "persz", "V-signed vs pers-Z")]

## Corrected path (V12-1, V12-2 unchanged from v1.4)

In [ ]:
# V12-1 (v1.3: forward branch of smooth_target uses min_periods=k) Building blocks: imputation, causal targets, target support, purge
def _feature_matrix_nan(d0, feat_names):
    """Raw feature matrix with NaN kept (columns absent from the frame -> 0.0, as in v1)."""
    n = len(d0)
    cols = [pd.to_numeric(d0[c], errors="coerce").values if c in d0.columns else np.zeros(n)
            for c in feat_names]
    return np.column_stack(cols).astype(float)

def impute_from_rows(X_nan, row_mask):
    """Fill NaN with the column median over row_mask only (column all-NaN there -> 0.0)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(X_nan[row_mask], axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    return np.where(np.isnan(X_nan), med, X_nan)

def impute_whole_series_v1(d0, feat_names):
    """Exact v1 behaviour: whole-series median per column."""
    df = d0.copy()
    for col in feat_names:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = 0.0
    return df[feat_names].values

def impute_causal(X_nan, B):
    """Row d filled with the median of rows [0, max(d+1, B)). Causal after the burn-in."""
    n = len(X_nan); out = X_nan.copy()
    for d in range(n):
        if np.isnan(X_nan[d]).any():
            L = min(max(d + 1, B), n)
            m = np.zeros(n, bool); m[:L] = True
            out[d] = impute_from_rows(X_nan, m)[d]
    return out

def smooth_target(idx, alignment, k=K):
    s = pd.Series(np.asarray(idx, float))
    if alignment == "trailing_v1":      # y[s] = mean(idx[s-k .. s-1])  (v1 convention)
        return s.rolling(k, min_periods=1).mean().shift(1).values
    if alignment == "forward":          # y[s] = mean(idx[s+1 .. s+k])
        # v1.3: full window required (min_periods=k). Rows with fewer than k following rows get NaN
        # and are dropped from scoring; with min_periods=1 they averaged fewer than 7 days.
        return s[::-1].rolling(k, min_periods=k).mean()[::-1].shift(-1).values
    raise ValueError(alignment)

def target_support(s, alignment, n, k=K):
    """Days whose index values enter y[s]."""
    if alignment == "trailing_v1":
        return set(range(max(0, s - k), s))
    return set(range(s + 1, min(n, s + k + 1)))

def purge_training_rows(tr_idx, te_idx, alignment, n):
    """Drop training rows whose target window touches the test day or its target window,
    or that lie inside the test day's target window."""
    test_days = set(int(t) for t in te_idx)
    test_support = set().union(*[target_support(int(s), alignment, n) for s in te_idx])
    blocked = test_days | test_support
    keep = [int(t) for t in tr_idx
            if int(t) not in test_support and not (target_support(int(t), alignment, n) & blocked)]
    return np.array(keep, dtype=int), len(tr_idx) - len(keep)

In [ ]:
# V12-2 Participant preparation, causal channels, the unified runner
_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE = {}, {}, {}
RUN_LOGS = {}   # label -> per-participant fold log (burn-in used, rows purged, first fold days)

def prepare(p, feat_names, has_labels, participants):
    key = (p["pid"], tuple(feat_names), bool(has_labels))
    if key in _PREP_CACHE:
        return _PREP_CACHE[key]
    d0 = p["days_df"].copy().reset_index(drop=True)
    n = len(d0)
    X_nan = _feature_matrix_nan(d0, feat_names)
    X_ws = impute_whole_series_v1(d0, feat_names)
    pm, pmad, _ = _compute_population_volatility_norm(participants, p["pid"], feat_names, has_labels)
    full = np.ones(n, bool)
    T_ws = ndi.compute_T_v3(X_ws, full)
    mi, ii = ndi.compute_quality_signal(d0, feat_names, full)
    I_ws = ndi.compute_I_v3(mi, ii)
    F_ws = ndi.compute_F_v3(X_ws, feat_names, full, pm, pmad)
    out = dict(pid=p["pid"], d0=d0, n=n, X_nan=X_nan, X_ws=X_ws, pm=pm, pmad=pmad,
               T_ws=T_ws, I_ws=I_ws, F_ws=F_ws, feat_names=list(feat_names),
               t0=min(10, max(5, n // 3)))
    _PREP_CACHE[key] = out
    return out

def get_folds(pr, g):
    fi = ndi.blocked_tscv(pr["X_ws"], np.zeros(pr["n"]), ndi.rf_factory,
                          T0=pr["t0"], g=g, return_folds=True)
    return [(np.array(a, int), np.array(b, int)) for a, b in fi.get("folds", [])]

def causal_channels(pr, B):
    """T, I, F where day d is computed from rows [0, max(d+1, B)) only."""
    key = (pr["pid"], tuple(pr["feat_names"]), B)
    if key in _CAUSAL_CACHE:
        return _CAUSAL_CACHE[key]
    n, fn = pr["n"], pr["feat_names"]
    Xc = impute_causal(pr["X_nan"], B)
    T, I, F = (np.full(n, np.nan) for _ in range(3))
    for d in range(n):
        L = min(max(d + 1, B), n); ones = np.ones(L, bool)
        T[d] = ndi.compute_T_v3(Xc[:L], ones)[d]
        F[d] = ndi.compute_F_v3(Xc[:L], fn, ones, pr["pm"], pr["pmad"])[d]
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, ones)
        I[d] = ndi.compute_I_v3(mi, ii)[d]
    _CAUSAL_CACHE[key] = (T, I, F, Xc)
    return _CAUSAL_CACHE[key]

def build_target(pr, target, target_stats, alignment, lam, B):
    """Returns (y, index_series_or_None)."""
    n = pr["n"]
    if target_stats == "whole_series_v1":
        if target == "self":
            idx = ndi.compute_ndi_v3_primary(pr["T_ws"], pr["I_ws"], pr["F_ws"], lambda_vec=tuple(lam))
            y = pd.Series(idx).rolling(K, min_periods=1).mean().shift(1).fillna(0).values
            return y, np.asarray(idx, float)
        X = pr["X_ws"]; comp = ((X - X.mean(0)) / (X.std(0) + 1e-8)).mean(1)
        return pd.Series(comp).rolling(K, min_periods=1).mean().shift(1).values, None
    T, I, F, Xc = causal_channels(pr, B)
    if target == "self":
        idx = np.asarray(ndi.compute_ndi_v3_primary(T, I, F, lambda_vec=tuple(lam)), float)
        return smooth_target(idx, alignment), idx
    comp = np.full(n, np.nan)
    for d in range(n):
        L = min(max(d + 1, B), n)
        mu, sd = Xc[:L].mean(0), Xc[:L].std(0) + 1e-8
        comp[d] = ((Xc[d] - mu) / sd).mean()
    return smooth_target(comp, alignment), None

def fold_features(pr, tr_idx, te_idx, whole_series, need_vsigned):
    fn = pr["feat_names"]
    if whole_series:                                  # v1 feature path (= leak L1)
        n = pr["n"]; tm = np.zeros(n, bool); tm[tr_idx] = True
        X = pr["X_ws"]
        X3 = np.column_stack([pr["T_ws"], pr["I_ws"], pr["F_ws"]])
    else:                                             # corrected: training stats, truncated at test day
        L = int(te_idx.max()) + 1
        tm = np.zeros(L, bool); tm[tr_idx] = True
        X = impute_from_rows(pr["X_nan"][:L], tm)
        T = ndi.compute_T_v3(X, tm)
        mi, ii = ndi.compute_quality_signal(pr["d0"].iloc[:L].reset_index(drop=True), fn, tm)
        I = ndi.compute_I_v3(mi, ii)
        F = ndi.compute_F_v3(X, fn, tm, pr["pm"], pr["pmad"])
        X3 = np.column_stack([T, I, F])
    XVS = None
    if need_vsigned:
        vs = ndi.build_neutrosophic_features(X, fn, train_mask=tm)
        XVS = vs[["Depression_T", "Depression_I", "Depression_F"]].fillna(0).values
    return X, X3, XVS

def _zscore_ridge(X_tr, y_tr, X_te):
    mu = X_tr.mean(0); sg = X_tr.std(0) + 1e-8
    return Ridge(alpha=ndi.CONFIG["RIDGE_ALPHA"]).fit((X_tr - mu) / sg, y_tr).predict((X_te - mu) / sg)

def run_arms(participants, feat_names, has_labels, target="self", arms=ARMS,
             target_stats="causal", alignment=None, whole_series_features=False,
             g=G, purge=None, lam=None, burn_in=None, target_mode="fixed", label=""):
    """One runner for Table 1, the raw-target comparison and every Task 2 state.
    target: "self" | "raw".  target_mode: "fixed" (primary) | "per_fold" (sensitivity row)."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    purge = V1_2_CONFIG["PURGE_SUPPORT_OVERLAP"] if purge is None else purge
    lam = tuple(lam or V1_2_CONFIG["FIXED_LAMBDA"])
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    v1_mode = (target_stats == "whole_series_v1")
    rows, log, t_start = [], [], time.time()
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        n = pr["n"]
        if n < 15:
            continue
        try:
            folds = get_folds(pr, g)
        except Exception:
            continue
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)   # burn-in inside every training window
        key = (pr["pid"], tuple(feat_names), target, target_stats, alignment, whole_series_features,
               g, purge, lam, B, target_mode)
        cached = _RUN_CACHE.get(key)
        if cached is not None and set(arms) <= set(cached["preds"]):
            res = cached
        else:
            y_fixed, idx = build_target(pr, target, "whole_series_v1" if v1_mode else "causal",
                                        alignment, lam, B)
            preds = {a: [] for a in arms}; y_all, te_all, n_purged = [], [], 0
            for tr_idx, te_idx in folds:
                if int(te_idx.min()) - int(tr_idx.max()) < g:
                    continue
                if purge and not v1_mode:
                    tr_idx, npg = purge_training_rows(tr_idx, te_idx, alignment, n); n_purged += npg
                Xf, X3, XVS = fold_features(pr, tr_idx, te_idx, whole_series_features or v1_mode,
                                            need_vsigned="vsigned" in arms)
                if target_mode == "per_fold":
                    assert target == "self" and alignment == "trailing_v1", "per-fold rebuild: trailing self-target only"
                    idx_f = ndi.compute_ndi_v3_primary(X3[:, 0], X3[:, 1], X3[:, 2], lambda_vec=lam)
                    y = smooth_target(idx_f, alignment)
                else:
                    y = y_fixed
                tr_v = tr_idx[np.isfinite(y[tr_idx])]; te_v = te_idx[np.isfinite(y[te_idx])]
                if len(tr_v) < 5 or len(te_v) < 1:
                    continue
                try:
                    fold_pred = {}
                    if "vsigned" in arms:
                        m = ndi.rf_factory(); m.fit(XVS[tr_v], y[tr_v]); fold_pred["vsigned"] = m.predict(XVS[te_v])
                    if "v3" in arms:
                        m = ndi.rf_factory(); m.fit(X3[tr_v], y[tr_v]); fold_pred["v3"] = m.predict(X3[te_v])
                    if "rawrf" in arms:
                        m = ndi.rf_factory(); m.fit(Xf[tr_v], y[tr_v]); fold_pred["rawrf"] = m.predict(Xf[te_v])
                    if "persz" in arms:
                        fold_pred["persz"] = _zscore_ridge(Xf[tr_v], y[tr_v], Xf[te_v])
                except Exception:
                    continue
                for a in arms:
                    preds[a].extend(np.asarray(fold_pred[a], float).tolist())
                y_all.extend(y[te_v].tolist()); te_all.extend(te_v.tolist())
            res = dict(preds=preds, y=y_all, te=te_all, idx=idx, y_fixed=y_fixed, B=B, n_purged=n_purged)
            _RUN_CACHE[key] = res
        if len(res["y"]) < 3:
            continue
        yt = np.array(res["y"]); row = dict(pid=pr["pid"], n_folds=len(yt))
        for a in arms:
            pa = np.array(res["preds"][a]); ok = np.isfinite(yt) & np.isfinite(pa)
            row[a] = spearmanr(yt[ok], pa[ok])[0] if ok.sum() >= 3 else np.nan
        rows.append(row)
        log.append(dict(pid=pr["pid"], burn_in_used=res["B"], purged_rows_summed_over_folds=res["n_purged"],
                        first_train_max=int(folds[0][0].max()), first_test_day=int(folds[0][1].min())))
    out = pd.DataFrame(rows)
    RUN_LOGS[label or target] = pd.DataFrame(log)
    print(f"  [{label or target}] {len(out)} participants, {time.time() - t_start:.0f}s")
    return out

def l2_proxy_rho(participants, feat_names, has_labels, lam, alignment=None, g=G, burn_in=None):
    """Leak L2: no model -- target index at the most recent available day vs the target,
    on exactly the test days the model-based runner evaluated."""
    alignment = alignment or V1_2_CONFIG["TARGET_ALIGNMENT"]
    burn_in = burn_in or V1_2_CONFIG["TARGET_BURN_IN_DAYS"]
    rows = []
    for p in participants:
        if p.get("days_df") is None:
            continue
        pr = prepare(p, feat_names, has_labels, participants)
        if pr["n"] < 15:
            continue
        folds = get_folds(pr, g)
        if not folds:
            continue
        B = min(burn_in, int(min(int(tr.max()) for tr, _ in folds)) + 1)
        y, idx = build_target(pr, "self", "causal", alignment, tuple(lam), B)
        days = np.array(sorted({int(s) for _, te in folds for s in te}))
        lag = 1 if alignment == "trailing_v1" else 0
        days = days[days >= lag]
        sc, tg = idx[days - lag], y[days]
        ok = np.isfinite(sc) & np.isfinite(tg)
        if ok.sum() >= 3:
            rows.append(dict(pid=pr["pid"], n_folds=int(ok.sum()), v3=spearmanr(sc[ok], tg[ok])[0]))
    return pd.DataFrame(rows)

## Task 3

In [ ]:
# T3-1 Task 3 targets behind flags: target "mood" / "stress" with alignment "next_day".
# build_target and target_support are wrapped, not edited: every v1.4 target and alignment goes to the v1.4 code
# unchanged (checked in T3-R). run_arms, folds, purge, features, arms and models are the V12-2 code as run in v1.4.
_need = ["build_target", "target_support", "run_arms", "wellness_daily", "load_wellness_reports", "W_ITEMS_1_5",
         "DATE_CHECK", "TASK3_CONFIG"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the cells above first. Missing: {_miss}"
TASK3_TARGETS = ("mood", "stress")
if "_build_target_v14" not in globals():            # wrap once, even if this cell is re-run
    _build_target_v14, _target_support_v14 = build_target, target_support

def target_support(s, alignment, n, k=K):
    """v1.4 rule for 'trailing_v1' / 'forward'; 'next_day': row s is scored against day s + 1 only."""
    if alignment == "next_day":
        return {s + 1}
    return _target_support_v14(s, alignment, n, k)

def build_target(pr, target, target_stats, alignment, lam, B):
    """Task 3 targets: the raw W1-W4 value of the next calendar day (NaN if there is none). Others: v1.4 code."""
    if target in TASK3_TARGETS:
        assert alignment == "next_day", "Task 3 targets are next-day values: call run_arms(..., alignment='next_day')"
        return TASK3_Y[(pr["pid"], target)].copy(), None
    return _build_target_v14(pr, target, target_stats, alignment, lam, B)

# --- targets: y[s] = value on local day (date of row s) + 1, from the W1-W4 table; rows are consecutive days (D-2)
_wd = wellness_daily.copy(); _wd["date"] = pd.to_datetime(_wd["date"])
TASK3_Y = {}
for p in pmdata_participants:
    d = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)
    w = _wd[_wd["pid"] == p["pid"]].set_index("date")
    for t in TASK3_TARGETS:
        TASK3_Y[(p["pid"], t)] = (d + pd.Timedelta(days=1)).map(w[t]).astype(float).values

# --- feature check: no wellness.csv field enters any arm's features
_well_fields = set()
for _pid in sorted(x for x in os.listdir(PMDATA_DIR) if x.startswith("p")):
    _wp = os.path.join(PMDATA_DIR, _pid, "pmsys", "wellness.csv")
    if os.path.isfile(_wp):
        _well_fields |= {c.strip().lower().replace(" ", "_") for c in pd.read_csv(_wp, nrows=0).columns}
assert not (set(PMDATA_FEATURE_NAMES) & _well_fields), "a wellness.csv field is a model feature -- STOP"
assert (DATE_CHECK["wellness_columns_in_days_df"] == 0).all(), "wellness columns in days_df -- STOP"
FEATURE_CHECK = (f"No wellness.csv field enters any arm's features: the {len(PMDATA_FEATURE_NAMES)} features "
                 f"({', '.join(PMDATA_FEATURE_NAMES)}) come from the Fitbit files only; wellness.csv fields "
                 f"({', '.join(sorted(_well_fields))}) are not in days_df (D-2) and are used only to build the "
                 "Task 3 target.")
print(FEATURE_CHECK)

# --- per-day W4 information for the log: valid values of each item that local day, after W3 and W2
W4_DAY = {}
for p in pmdata_participants:
    rep = load_wellness_reports(PMDATA_DIR, p["pid"])
    if rep is None or rep.empty:
        continue
    r = rep[rep["ts_utc"].notna() & rep["tz_aware"]].copy()
    r = r[~(r[W_ITEMS_1_5] == 0).all(axis=1)].copy()                    # W3
    for c in W_ITEMS_1_5:
        r.loc[r[c] == 0, c] = np.nan                                    # W2
    g = r.groupby("local_date")
    for t in TASK3_TARGETS:
        W4_DAY[(p["pid"], t)] = (g[t].count(), g[t].nunique())
for t in TASK3_TARGETS:
    n_days = sum(int(np.isfinite(v).sum()) for (pid, tt), v in TASK3_Y.items() if tt == t)
    print(f"{t}: {n_days} participant-rows have a next-day value")

In [ ]:
# T3-R Reproduction check (run first, and again last in T3-H): with the Task 3 wrappers in place, the v1.4 corrected
# Table 1 and raw-feature comparison reproduce results_v1.4 exactly, and the wellness table equals v1.4's.
# About 8 minutes on Colab.
import io
def reproduce_v14(label):
    for _c in (_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE):
        _c.clear()
    worst_all = {}
    for tgt, fname in [("self", "table1_corrected_self_target_per_participant.csv"),
                       ("raw", "raw_target_corrected_per_participant.csv")]:
        new = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target=tgt, label=f"{label} {tgt}")
        new = new[new["n_folds"] >= MF].set_index("pid")
        ref = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, fname)).set_index("pid")
        assert sorted(new.index) == sorted(ref.index), f"{tgt}: participant set differs from v1.4 -- STOP"
        assert (new.loc[ref.index, "n_folds"] == ref["n_folds"]).all(), f"{tgt}: n_folds differ from v1.4 -- STOP"
        worst = max(float((new.loc[ref.index, a] - ref[a]).abs().max()) for a in ARMS)
        assert worst < 1e-9, f"{tgt}: max |diff| vs results_v1.4 = {worst:.2e} -- STOP"
        worst_all[tgt] = worst
        print(f"  {tgt}: {len(ref)} participants reproduce results_v1.4 (max |diff| = {worst:.1e})")
    return worst_all

REPRO_FIRST = reproduce_v14("reproduce v1.4 (first)")
_wa = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "wellness_daily_W1_W4.csv"))
_ws = pd.read_csv(io.StringIO(wellness_daily.to_csv(index=False)))      # same CSV round trip as the archived file
pd.testing.assert_frame_equal(_ws.sort_values(["pid", "date"]).reset_index(drop=True),
                              _wa.sort_values(["pid", "date"]).reset_index(drop=True), check_dtype=False)
WELLNESS_MATCHES_V14 = True
print(f"  wellness table: {len(_wa)} rows, identical to results_v1.4/wellness_daily_W1_W4.csv")
print("REPRODUCTION CHECK (first) PASSED")

In [ ]:
# T3-2 Task 3 runs: next-day mood (primary) and next-day stress (secondary), all four arms, then eligibility.
# v1.4 review memo, Section 4 item 2: every eligible participant must have a finite rho in all four arms.
# About 4 minutes per target on Colab.
T3_ALL, T3_ELIG, _elog = {}, {}, []
def _cached(pid, tgt, cache=None):
    for k, v in (cache if cache is not None else _RUN_CACHE).items():
        if k[0] == pid and k[2] == tgt and k[4] == "next_day":
            return v
    return None

for tgt in TASK3_TARGETS:
    T3_ALL[tgt] = run_arms(pmdata_participants, PMDATA_FEATURE_NAMES, False, target=tgt, alignment="next_day",
                           label=f"task3 {tgt}")
    _pl = RUN_LOGS[f"task3 {tgt}"]
    assert int(_pl["purged_rows_summed_over_folds"].sum()) == 0, "next-day purge removed training rows -- check"
    for p in pmdata_participants:
        pid, n = p["pid"], len(p["days_df"])
        t0 = min(10, max(5, n // 3)); n_test = max(0, n - (t0 + G))
        res = _cached(pid, tgt)
        yv = np.asarray(res["y"], float) if res else np.array([])
        te = np.asarray(res["te"], int) if res else np.array([], int)
        n_sc, n_dist = len(yv), len(np.unique(yv))
        dates = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)
        cnt, nun = W4_DAY.get((pid, tgt), (pd.Series(dtype=int), pd.Series(dtype=int)))
        tdays = [(dates[s] + pd.Timedelta(days=1)).strftime("%Y-%m-%d") for s in te]
        w4_choice = int(sum(cnt.get(d, 0) >= 2 for d in tdays)); w4_diff = int(sum(nun.get(d, 0) >= 2 for d in tdays))
        if n_sc >= MF and n_dist >= 3:
            reason = ""
        elif n_sc < MF:
            reason = f"fewer than {MF} scored test folds ({n_sc})"
        else:
            reason = f"fewer than 3 distinct {tgt} values on the scored test days ({n_dist}: {sorted(set(yv.tolist()))})"
        _elog.append(dict(target=tgt, pid=pid, rows=n, test_rows=n_test,
                          rows_with_next_day_value=int(np.isfinite(TASK3_Y[(pid, tgt)]).sum()),
                          scored_test_folds=n_sc, test_days_without_next_day_value=n_test - n_sc,
                          distinct_values_on_scored_days=n_dist,
                          scored_days_where_W4_chose_among_2plus_values=w4_choice,
                          scored_days_where_those_values_differed=w4_diff,
                          eligible=reason == "", exclusion_reason=reason))
T3_ELIGLOG = pd.DataFrame(_elog)
for tgt in TASK3_TARGETS:
    ok = set(T3_ELIGLOG[(T3_ELIGLOG["target"] == tgt) & T3_ELIGLOG["eligible"]]["pid"])
    T3_ELIG[tgt] = T3_ALL[tgt][T3_ALL[tgt]["pid"].isin(ok)].reset_index(drop=True)
    _nan = [(r["pid"], a) for _, r in T3_ELIG[tgt].iterrows() for a in ARMS if not np.isfinite(r[a])]
    if _nan:
        raise RuntimeError(f"STOP ({tgt}): eligible participants with a missing rho, so paired tests would use "
                           f"different n: {_nan}. Report this to ma'am (v1.4 review memo, Section 4 item 2).")
    _sc = T3_ELIGLOG[T3_ELIGLOG["target"] == tgt].set_index("pid")["scored_test_folds"]
    assert all(int(_sc[r.pid]) == int(r.n_folds) for r in T3_ALL[tgt].itertuples()), "scored folds != n_folds"
    out = T3_ALL[tgt].merge(T3_ELIGLOG[T3_ELIGLOG["target"] == tgt][["pid", "eligible"]], on="pid", how="left")
    out.to_csv(os.path.join(RESULTS_T3_DIR, f"task3_{tgt}_per_participant.csv"), index=False)
    RUN_LOGS[f"task3 {tgt}"].to_csv(os.path.join(RESULTS_T3_DIR, f"task3_fold_log_{tgt}.csv"), index=False)
T3_ELIGLOG.to_csv(os.path.join(RESULTS_T3_DIR, "task3_eligibility_log.csv"), index=False)
pd.set_option("display.width", 250)
print(T3_ELIGLOG.to_string(index=False))
for tgt in TASK3_TARGETS:
    _e = T3_ELIGLOG[T3_ELIGLOG["target"] == tgt]
    print(f"{tgt}: {int(_e['eligible'].sum())} of {len(_e)} participants eligible; excluded: "
          + (", ".join(f"{r.pid} ({r.exclusion_reason})" for r in _e[~_e['eligible']].itertuples()) or "none")
          + f"; W4 chose among 2+ values on {int(_e['scored_days_where_W4_chose_among_2plus_values'].sum())} scored "
            f"target days (values differed on {int(_e['scored_days_where_those_values_differed'].sum())})")

In [ ]:
# T3-3 Task 3 statistics, identical to Task 1. The functions below are copied unchanged from V12-6 (v1.4).
# Changed by the v1.4 review memo (Section 4 item 1): the artifact bound uses the self-derived advantage of the same
# eligible participants; V3 vs V-signed has no artifact bound.
def _rng():
    return np.random.default_rng(SEED)

def _boot(x, stat, level):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3 or np.allclose(x, x[0]):
        return np.nan, np.nan, "degenerate"
    for method in ("BCa", "percentile"):
        try:
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, rng=_rng())
        except TypeError:   # scipy < 1.15
            r = scipy_bootstrap((x,), stat, n_resamples=N_BOOT, method=method,
                                confidence_level=level, vectorized=False, random_state=_rng())
        lo, hi = float(r.confidence_interval.low), float(r.confidence_interval.high)
        if np.isfinite(lo) and np.isfinite(hi):
            return lo, hi, method
    return np.nan, np.nan, "failed"

def hodges_lehmann(d):
    d = np.asarray(d, float); n = len(d)
    return float(np.median([(d[i] + d[j]) / 2 for i in range(n) for j in range(i, n)]))

def bh(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    q = np.empty(m); prev = 1.0
    for rank, i in reversed(list(enumerate(order, start=1))):
        prev = min(prev, p[i] * m / rank); q[i] = prev
    return q

def paired_row(df, a, b, label, target):
    d = df[[a, b]].dropna(); diff = (d[a] - d[b]).values
    lo95, hi95, m95 = _boot(diff, np.mean, 0.95)
    lo90, hi90, _ = _boot(diff, np.mean, 0.90)
    hlo, hhi, _ = _boot(diff, hodges_lehmann, 0.95)
    try:
        pv = float(wilcoxon(d[a], d[b]).pvalue)
    except ValueError:
        pv = np.nan
    return dict(comparison=label, target=target, n=len(d), mean_delta_rho=float(diff.mean()),
                ci95_lo=lo95, ci95_hi=hi95, ci_method=m95,
                hl_median=hodges_lehmann(diff), hl_ci95_lo=hlo, hl_ci95_hi=hhi,
                wins=int((diff > 0).sum()), p=pv, ci90_lo=lo90, ci90_hi=hi90)

# Artifact bound (v1.4 review memo, Section 4 item 1): self-derived advantage on the SAME eligible participants,
# from the v1.4 per-participant file (without-p12 rows: from the v1.4 without-p12 per-participant file).
SELF_PP_V14 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "table1_corrected_self_target_per_participant.csv"))
SELF_PP_V14_NOP12 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "supplementary_without_p12_self_target_per_participant.csv"))
NMIN = TASK3_CONFIG["MIN_ELIGIBLE_FOR_TESTS"]
_ND = TASK3_CONFIG["REPORTING_DECIMALS"]

def task3_stats(df, tgt_label, self_pp=None):
    """Task 1 statistics on one target. Fewer than NMIN participants: descriptive only (no P, q, TOST)."""
    self_pp = SELF_PP_V14 if self_pp is None else self_pp
    tests = len(df) >= NMIN
    block = [paired_row(df, a, b, lab, tgt_label) for a, b, lab in COMPARISONS]
    qs = bh([r["p"] for r in block]) if tests else [np.nan] * len(block)
    for r, q in zip(block, qs):
        r["q"] = q
    rows = block + [{**paired_row(df, "v3", "vsigned", "V3 vs V-signed", tgt_label), "q": np.nan}]
    t2 = pd.DataFrame(rows)
    if tests:
        t2["tost_equivalent_at_SESOI"] = (t2["ci90_lo"] > -SESOI) & (t2["ci90_hi"] < SESOI)
        t2["advantage_ge_SESOI_rejected"] = t2["ci90_hi"] < SESOI
    else:
        t2["p"] = np.nan
        t2["tost_equivalent_at_SESOI"] = "n/a (descriptive)"; t2["advantage_ge_SESOI_rejected"] = "n/a (descriptive)"
    sp = self_pp[self_pp["pid"].isin(set(df["pid"]))]
    assert set(sp["pid"]) == set(df["pid"]), "eligible participants missing from the self-derived per-participant file"
    _adv = {}
    for a, b, lab in COMPARISONS:                     # V3 vs V-signed has no q: no artifact bound
        dd = (sp[a] - sp[b]).values
        lo, hi, _ = _boot(dd, np.mean, 0.95)
        _adv[lab] = (float(dd.mean()) if len(dd) else np.nan, lo, hi)
    t2["self_derived_advantage_same_participants"] = t2["comparison"].map(lambda c: _adv.get(c, (np.nan,) * 3)[0])
    t2["self_derived_advantage_ci95_lo"] = t2["comparison"].map(lambda c: _adv.get(c, (np.nan,) * 3)[1])
    t2["self_derived_advantage_ci95_hi"] = t2["comparison"].map(lambda c: _adv.get(c, (np.nan,) * 3)[2])
    t2["self_derived_advantage_n"] = len(sp)
    _ok = t2["self_derived_advantage_same_participants"].notna() & t2["ci95_hi"].notna()
    t2["artifact_excluded"] = np.where(_ok, t2["ci95_hi"] < t2["self_derived_advantage_same_participants"], "n/a")
    t2["analysis"] = "inferential" if tests else f"descriptive (fewer than {NMIN} eligible)"
    ab = []
    for a in ARMS:
        v = df[a].dropna().values; lo, hi, _ = _boot(v, np.mean, 0.95)
        try:
            p0 = float(wilcoxon(v).pvalue) if tests else np.nan
        except ValueError:
            p0 = np.nan
        ab.append(dict(arm=ARM_LABEL[a], target=tgt_label, n=len(v), mean_rho=float(v.mean()) if len(v) else np.nan,
                       ci95_lo=lo, ci95_hi=hi, p_vs_zero=p0))
    return t2, pd.DataFrame(ab)

def _f(x):
    return "n/a" if pd.isna(x) else f"{x:.{_ND}f}"
def _ci(m, l, h):
    return f"{_f(m)} [{_f(l)}, {_f(h)}]"
def _flag(v, yes, no):
    s = str(v).strip().lower()
    return yes if s == "true" else no if s == "false" else "n/a"
def format_t2(t2):
    return pd.DataFrame({
        "Comparison": t2["comparison"], "Target": t2["target"],
        "Mean delta rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(t2["mean_delta_rho"], t2["ci95_lo"], t2["ci95_hi"])],
        "HL median [95% CI]": [_ci(m, l, h) for m, l, h in zip(t2["hl_median"], t2["hl_ci95_lo"], t2["hl_ci95_hi"])],
        "Wins/n": [f"{w}/{n}" for w, n in zip(t2["wins"], t2["n"])],
        "P": [_f(v) for v in t2["p"]], "q (BH)": [_f(v) for v in t2["q"]],
        "TOST (+/-0.10), 90% CI": [("n/a" if _flag(e, "y", "n") == "n/a" else
                                   f"{_flag(e, 'equivalent', 'inconclusive')} [{_f(l)}, {_f(h)}]")
                                  for e, l, h in zip(t2["tost_equivalent_at_SESOI"], t2["ci90_lo"], t2["ci90_hi"])],
        "Advantage >= 0.10 rejected": [_flag(v, "yes", "no") for v in t2["advantage_ge_SESOI_rejected"]],
        "Artifact excluded": [_flag(v, "yes", "no") for v in t2["artifact_excluded"]]})
def format_abs(ab):
    return pd.DataFrame({"Arm": ab["arm"], "Target": ab["target"], "n": ab["n"],
                         "Mean rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(ab["mean_rho"], ab["ci95_lo"], ab["ci95_hi"])],
                         "P vs 0": [_f(v) for v in ab["p_vs_zero"]]})

T3_T2, T3_ABS = {}, {}
for tgt, lab in [("mood", "next-day mood"), ("stress", "next-day stress")]:
    T3_T2[tgt], T3_ABS[tgt] = task3_stats(T3_ELIG[tgt], lab)
    T3_T2[tgt].to_csv(os.path.join(RESULTS_T3_DIR, f"task3_table2_{tgt}.csv"), index=False)
    T3_ABS[tgt].to_csv(os.path.join(RESULTS_T3_DIR, f"task3_absolute_rho_{tgt}.csv"), index=False)
    format_t2(T3_T2[tgt]).to_csv(os.path.join(RESULTS_T3_DIR, f"task3_table2_{tgt}_formatted.csv"), index=False)
    format_abs(T3_ABS[tgt]).to_csv(os.path.join(RESULTS_T3_DIR, f"task3_absolute_rho_{tgt}_formatted.csv"), index=False)
    print(f"\n=== {lab}: n = {len(T3_ELIG[tgt])} eligible ({T3_T2[tgt]['analysis'].iloc[0]})")
    print(format_abs(T3_ABS[tgt]).to_string(index=False)); print(format_t2(T3_T2[tgt]).to_string(index=False))

In [ ]:
# T3-4 Task 3 without p12 (sensitivity analysis, as v1.4's supplementary row): for each target where p12 is eligible, the run is repeated
# with p12 removed from the cohort AND from every donor pool. Caches are set aside and restored, so no other cell
# sees a p12-free donor pool. About 4 minutes per target on Colab.
T3_NOP12, _np_notes = {}, []
_p12_elig = {t: bool(((T3_ELIGLOG["target"] == t) & (T3_ELIGLOG["pid"] == "p12") & T3_ELIGLOG["eligible"]).any())
             for t in TASK3_TARGETS}
_targets_np = [t for t in TASK3_TARGETS if _p12_elig[t]]
if _targets_np:
    _cohort = [p for p in pmdata_participants if p["pid"] != "p12"]
    _saved = [dict(c) for c in (_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE)]
    for _c in (_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE):
        _c.clear()
    try:
        for tgt in _targets_np:
            _all = run_arms(_cohort, PMDATA_FEATURE_NAMES, False, target=tgt, alignment="next_day",
                            label=f"task3 {tgt} without p12")
            _ok = set(T3_ELIG[tgt]["pid"]) - {"p12"}          # eligibility does not depend on the donor pools
            T3_NOP12[tgt] = _all[_all["pid"].isin(_ok)].reset_index(drop=True)
            _ref = T3_ELIG[tgt].set_index("pid")
            assert all(int(_ref.loc[r.pid, "n_folds"]) == int(r.n_folds) for r in T3_NOP12[tgt].itertuples())
            _nan = [(r["pid"], a) for _, r in T3_NOP12[tgt].iterrows() for a in ARMS if not np.isfinite(r[a])]
            if _nan:
                raise RuntimeError(f"STOP ({tgt}, without p12): missing rho {_nan} -- report to ma'am")
    finally:
        for _c, _sv in zip((_PREP_CACHE, _CAUSAL_CACHE, _RUN_CACHE), _saved):
            _c.clear(); _c.update(_sv)
    for tgt in _targets_np:
        lab = f"next-day {tgt}, without p12"
        t2, ab = task3_stats(T3_NOP12[tgt], lab, self_pp=SELF_PP_V14_NOP12)
        T3_NOP12[tgt].to_csv(os.path.join(RESULTS_T3_DIR, f"task3_{tgt}_without_p12_per_participant.csv"), index=False)
        t2.to_csv(os.path.join(RESULTS_T3_DIR, f"task3_table2_{tgt}_without_p12.csv"), index=False)
        ab.to_csv(os.path.join(RESULTS_T3_DIR, f"task3_absolute_rho_{tgt}_without_p12.csv"), index=False)
        format_t2(t2).to_csv(os.path.join(RESULTS_T3_DIR, f"task3_table2_{tgt}_without_p12_formatted.csv"), index=False)
        print(f"\n=== {lab}: n = {len(T3_NOP12[tgt])}")
        print(format_abs(ab).to_string(index=False)); print(format_t2(t2).to_string(index=False))
        _np_notes.append(f"{tgt}: p12 eligible; also reported without p12 (n = {len(T3_NOP12[tgt])}).")
for t in TASK3_TARGETS:
    if not _p12_elig[t]:
        _np_notes.append(f"{t}: p12 not eligible, so no without-p12 row is needed (p12 is still a donor for the others, as in v1.4).")
print("\n".join(_np_notes))
with open(os.path.join(RESULTS_T3_DIR, "task3_without_p12_notes.txt"), "w") as f:
    f.write("\n".join(_np_notes) + "\n")

In [ ]:
# T3-P Persistence references (v1.4 review memo, Section 4 items 3-4; fixed in config_task3_v1_4.json before the run).
# Reference rows only, outside the BH set. No model: today's value is the forecast.
#  (a) mood / stress: forecast for row s = the W1-W4 value of day d_s (today's report) for tomorrow's value,
#      on the Task 3 scored test days of the eligible participants where today's value exists;
#  (b) self-derived target: the L2 proxy (index today vs the forward target), all v1.4 participants; its mean must
#      equal the v1.4 "Clean + L2 only" value; paired V3 vs persistence, exploratory;
#  (c) raw-feature target: today's causal raw composite vs the forward raw-feature target, reference only.
_need = ["T3_ELIG", "TASK3_Y", "_cached", "l2_proxy_rho", "causal_channels", "smooth_target", "paired_row", "_boot"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run the cells above first. Missing: {_miss}"
def _mean_ci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    lo, hi, _ = _boot(v, np.mean, 0.95)
    return (float(v.mean()) if len(v) else np.nan), lo, hi, len(v)
_LAM = tuple(V1_2_CONFIG["FIXED_LAMBDA"])
_rows_pp, _rows_sum = [], []

# (a) next-day mood and stress
_wdP = wellness_daily.copy(); _wdP["date"] = pd.to_datetime(_wdP["date"])
for tgt in TASK3_TARGETS:
    for pid in T3_ELIG[tgt]["pid"]:
        p = next(q for q in pmdata_participants if q["pid"] == pid)
        d = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)
        today = d.map(_wdP[_wdP["pid"] == pid].set_index("date")[tgt]).astype(float).values
        res = _cached(pid, tgt); te = np.asarray(res["te"], int); y = TASK3_Y[(pid, tgt)]
        assert np.array_equal(np.asarray(res["y"], float), y[te]), f"{pid}: scored days differ from T3-2"
        ok = te[np.isfinite(today[te])]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rho = float(spearmanr(today[ok], y[ok])[0]) if len(ok) >= 3 else np.nan
        _rows_pp.append(dict(target=f"next-day {tgt}", pid=pid, scored_test_days=len(te), days_used=len(ok),
                             persistence_rho=rho))
# (b) self-derived target: L2 proxy, recomputed
_l2 = l2_proxy_rho(pmdata_participants, PMDATA_FEATURE_NAMES, False, _LAM, g=G)
_o14 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "task2_one_at_a_time.csv")).set_index("state")
L2_ARCHIVED = float(_o14.loc["Clean + L2 only", "v3_mean_rho"])
_t1_self = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "table1_corrected_self_target_per_participant.csv"))
_l2m = _l2[_l2["pid"].isin(_t1_self["pid"])]
L2_MATCH = abs(float(_l2m["v3"].mean()) - L2_ARCHIVED) < 1e-9
assert L2_MATCH, f"persistence mean {_l2m['v3'].mean():.6f} != v1.4 'Clean + L2 only' {L2_ARCHIVED:.6f} -- STOP"
for r in _l2m.itertuples():
    _rows_pp.append(dict(target="self-derived (v1.4)", pid=r.pid, scored_test_days=int(r.n_folds), days_used=int(r.n_folds),
                         persistence_rho=float(r.v3)))
# (c) raw-feature target: today's causal raw composite (the quantity the target smooths), same code as build_target
_t1_raw = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "raw_target_corrected_per_participant.csv"))
RAW_COMPOSITE_MATCH = True
for p in pmdata_participants:
    if p["pid"] not in set(_t1_raw["pid"]):
        continue
    pr = prepare(p, PMDATA_FEATURE_NAMES, False, pmdata_participants); n = pr["n"]
    folds = get_folds(pr, G)
    B = min(V1_2_CONFIG["TARGET_BURN_IN_DAYS"], int(min(int(tr.max()) for tr, _ in folds)) + 1)
    _, _, _, Xc = causal_channels(pr, B)
    comp = np.full(n, np.nan)
    for dd in range(n):
        L = min(max(dd + 1, B), n)
        mu, sd = Xc[:L].mean(0), Xc[:L].std(0) + 1e-8
        comp[dd] = ((Xc[dd] - mu) / sd).mean()
    y = smooth_target(comp, V1_2_CONFIG["TARGET_ALIGNMENT"])
    y_ref, _ = build_target(pr, "raw", "causal", V1_2_CONFIG["TARGET_ALIGNMENT"], _LAM, B)
    RAW_COMPOSITE_MATCH &= bool(np.array_equal(np.nan_to_num(y, nan=-9e9), np.nan_to_num(y_ref, nan=-9e9)))
    days = np.array(sorted({int(s) for _, te in folds for s in te}))
    ok = days[np.isfinite(comp[days]) & np.isfinite(y[days])]
    _rows_pp.append(dict(target="raw-feature (v1.4)", pid=p["pid"], scored_test_days=int(len(ok)), days_used=int(len(ok)),
                         persistence_rho=float(spearmanr(comp[ok], y[ok])[0])))
assert RAW_COMPOSITE_MATCH, "today's raw composite does not rebuild the v1.4 raw-feature target -- STOP"

PERSIST_PP = pd.DataFrame(_rows_pp)
for tgt, g_ in PERSIST_PP.groupby("target", sort=False):
    m, lo, hi, _kk = _mean_ci(g_["persistence_rho"])
    _rows_sum.append(dict(target=tgt, reference="persistence (today's value), no model", participants=_kk,
                          participants_without_rho=int(g_["persistence_rho"].isna().sum()),
                          days_used=int(g_["days_used"].sum()), mean_rho=m, ci95_lo=lo, ci95_hi=hi))
PERSIST_SUM = pd.DataFrame(_rows_sum)
# paired V3 vs persistence on the self-derived target (exploratory)
_pv = _t1_self[["pid", "v3"]].merge(_l2m[["pid", "v3"]].rename(columns={"v3": "persistence"}), on="pid")
V3_VS_PERSIST = pd.DataFrame([paired_row(_pv, "v3", "persistence", "V3 vs persistence (exploratory)", "self-derived")])
PERSIST_PP.to_csv(os.path.join(RESULTS_T3_DIR, "persistence_reference_per_participant.csv"), index=False)
PERSIST_SUM.to_csv(os.path.join(RESULTS_T3_DIR, "persistence_reference_summary.csv"), index=False)
V3_VS_PERSIST.to_csv(os.path.join(RESULTS_T3_DIR, "v3_vs_persistence_self_derived_exploratory.csv"), index=False)
pd.DataFrame({"Target": PERSIST_SUM["target"], "Participants": PERSIST_SUM["participants"],
              "Days used": PERSIST_SUM["days_used"],
              "Persistence mean rho [95% CI]": [_ci(m, l, h) for m, l, h in zip(PERSIST_SUM["mean_rho"], PERSIST_SUM["ci95_lo"], PERSIST_SUM["ci95_hi"])]}
             ).to_csv(os.path.join(RESULTS_T3_DIR, "persistence_reference_summary_formatted.csv"), index=False)
print(PERSIST_SUM.round(3).to_string(index=False))
r = V3_VS_PERSIST.iloc[0]
print(f"\nEXPLORATORY, self-derived target, n = {int(r['n'])}: V3 minus persistence = "
      f"{_ci(r['mean_delta_rho'], r['ci95_lo'], r['ci95_hi'])}, wins {int(r['wins'])}/{int(r['n'])}, Wilcoxon P = {_f(r['p'])}")
print(f"Persistence on the self-derived target matches v1.4 'Clean + L2 only' ({L2_ARCHIVED:.3f}): {L2_MATCH}; "
      f"raw composite rebuilds the v1.4 raw-feature target exactly: {RAW_COMPOSITE_MATCH}")

## Reporting items from v1.4 (v1.4 review memo, Section 4 item 5)

In [ ]:
# T3-7 Reporting items from v1.4 (v1.4 review memo, Section 4 item 5, and Sections 1-2). Read-only on results_v1.4;
# everything is written to results_v1.4_task3.
#  (a) target-swap shift without p12 (guide's values: V3 vs raw-RF 0.173 [0.082, 0.265], V3 vs pers-Z 0.157
#      [0.049, 0.280], both q = .025);
#  (b) Figure 4 redrawn from the L2-removed state, plain labels;
#  (c) Table 4 standalone inflation, with L2 under both targets (v1.2 trailing: leak; v1.4 forward: persistence);
#  (d) formatted Table 2 with "artifact excluded" n/a for V3 vs V-signed;
#  (e) reporting notes.
import matplotlib.pyplot as plt
# (a)
_ss = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "supplementary_without_p12_self_target_per_participant.csv"))
_sr = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "supplementary_without_p12_raw_target_per_participant.csv"))
_ms = _ss[["pid"] + ARMS].merge(_sr[["pid"] + ARMS], on="pid", suffixes=("_self", "_raw"))
_sw = []
for a, b, lab in COMPARISONS:
    d = ((_ms[f"{a}_self"] - _ms[f"{b}_self"]) - (_ms[f"{a}_raw"] - _ms[f"{b}_raw"])).values
    d = d[np.isfinite(d)]; lo, hi, meth = _boot(d, np.mean, 0.95)
    try:
        pv = float(wilcoxon(d).pvalue)
    except ValueError:
        pv = np.nan
    _sw.append(dict(comparison=lab, n=len(d), mean_shift=float(d.mean()), ci95_lo=lo, ci95_hi=hi, ci_method=meth,
                    n_positive=int((d > 0).sum()), p=pv))
SWAP_NOP12 = pd.DataFrame(_sw); SWAP_NOP12["q"] = bh(SWAP_NOP12["p"].values)
_GUIDE = {"V3 vs raw-RF": (0.173, 0.082, 0.265, 0.025), "V3 vs pers-Z": (0.157, 0.049, 0.280, 0.025)}
_cmp = []
for lab, (gm, glo, ghi, gq) in _GUIDE.items():
    r = SWAP_NOP12.set_index("comparison").loc[lab]
    _cmp.append(dict(comparison=lab, mean_matches=round(r["mean_shift"], 3) == gm, q_matches=round(r["q"], 3) == gq,
                     ci_matches=(round(r["ci95_lo"], 3), round(r["ci95_hi"], 3)) == (glo, ghi)))
SWAP_NOP12_GUIDE = pd.DataFrame(_cmp)
SWAP_NOP12_MATCH = bool(SWAP_NOP12_GUIDE["mean_matches"].all() and SWAP_NOP12_GUIDE["q_matches"].all())
SWAP_NOP12.to_csv(os.path.join(RESULTS_T3_DIR, "exploratory_target_swap_shift_without_p12.csv"), index=False)
pd.DataFrame({"Comparison": SWAP_NOP12["comparison"], "n": SWAP_NOP12["n"],
              "Mean shift [95% CI]": [_ci(m, l, h) for m, l, h in zip(SWAP_NOP12["mean_shift"], SWAP_NOP12["ci95_lo"], SWAP_NOP12["ci95_hi"])],
              "Positive/n": [f"{a}/{b}" for a, b in zip(SWAP_NOP12["n_positive"], SWAP_NOP12["n"])],
              "P": [_f(v) for v in SWAP_NOP12["p"]], "q (BH)": [_f(v) for v in SWAP_NOP12["q"]]}
             ).to_csv(os.path.join(RESULTS_T3_DIR, "exploratory_target_swap_shift_without_p12_formatted.csv"), index=False)
print("(a) EXPLORATORY target-swap shift WITHOUT p12 (sensitivity)"); print(SWAP_NOP12.round(3).to_string(index=False))
print(SWAP_NOP12_GUIDE.to_string(index=False)); print(f"    matches ma'am's values (mean and q): {SWAP_NOP12_MATCH}")

# (b) Figure 4 from the L2-removed state
_cum = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "task2_waterfall_cumulative.csv"))
_chain = _cum[_cum["removed"].isin(["L2", "L4", "L3", "L1", "L0"])].reset_index(drop=True)
assert list(_chain["removed"]) == ["L2", "L4", "L3", "L1", "L0"], "unexpected order in task2_waterfall_cumulative.csv"
_PLAIN = {"L0": "no embargo (L0)", "L1": "whole-series normalisation (L1)", "L3": "in-sample weights (L3)",
          "L4": "tuned-weight target (L4)"}
_labels = ["Start: four leaks on\n(L0, L1, L3, L4)"] + [f"− {_PLAIN[r]}" for r in _chain["removed"].iloc[1:]]
_labels[-1] = f"Leak-free\n(− {_PLAIN['L0']})"
_vals = [round(v, 3) for v in _chain["v3_mean_rho"]]
print(f"\n(b) Figure 4 chain: {' -> '.join(f'{v:.3f}' for v in _vals)}")
fig, ax = plt.subplots(figsize=(9, 4.8))
_x = np.arange(len(_chain))
_col = ["#C44E52"] + ["#BBBBBB" if r == "L3" else "#4C72B0" for r in _chain["removed"].iloc[1:]]
ax.bar(_x, _chain["v3_mean_rho"], color=_col)
ax.errorbar(_x, _chain["v3_mean_rho"], yerr=[_chain["v3_mean_rho"] - _chain["v3_ci95_lo"],
                                             _chain["v3_ci95_hi"] - _chain["v3_mean_rho"]], fmt="none", ecolor="black", capsize=4)
_clean = float(_chain["v3_mean_rho"].iloc[-1])
ax.axhline(_clean, ls="--", lw=1, color="crimson", label=f"Leak-free estimate, mean ρ = {_clean:.3f}")
ax.set_xticks(_x); ax.set_xticklabels(_labels, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("V3 mean within-participant ρ (PMData, ≥10 test folds)")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=8)
ax.set_ylim(0, max(0.9, float(_chain["v3_ci95_hi"].max()) + 0.05)); fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_T3_DIR, f"figure4_leak_waterfall_from_L2_removed.{ext}"), dpi=300)
plt.show()
pd.DataFrame({"bar": _labels, "state": _chain["state"], "v3_mean_rho": _chain["v3_mean_rho"].round(3),
              "ci95": [f"[{l:.3f}, {h:.3f}]" for l, h in zip(_chain["v3_ci95_lo"], _chain["v3_ci95_hi"])],
              "inert": _chain["removed"].eq("L3")}).to_csv(
    os.path.join(RESULTS_T3_DIR, "figure4_from_L2_removed_bar_labels.csv"), index=False)

# (c) Table 4: standalone inflation, L2 under both targets
_t4_14 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "task2_table4_inflation_column.csv")).set_index("leak")
_t4_12 = pd.read_csv(os.path.join(RESULTS_V1_2_ARCHIVE, "task2_table4_inflation_column.csv")).set_index("leak")
_names = {"L0": "No embargo (L0) -- literature-known reference leak", "L1": "Whole-series normalisation (L1)",
          "L2": "Self-predictive proxy (L2)", "L3": "In-sample weight selection (L3) -- inert for this arm",
          "L4": "Target built with tuned weights (L4)"}
_t4 = []
for lk in ["L0", "L1", "L2", "L3", "L4"]:
    row = dict(leak=_names[lk], standalone_inflation_v1_4_forward=float(_t4_14.loc[lk, "standalone_inflation"]))
    if lk == "L2":
        row["standalone_inflation_v1_2_trailing"] = float(_t4_12.loc[lk, "standalone_inflation"])
        row["note"] = ("Trailing target (v1.2): today's index lies inside the target window -- a leak. Forward target "
                       f"(v1.4): a naive persistence forecast, not a leak (rho {L2_ARCHIVED:.3f} vs V3 "
                       f"{float(_t1_self['v3'].mean()):.3f}); reported as a reference row.")
    else:
        row["standalone_inflation_v1_2_trailing"] = np.nan
        row["note"] = ""
    _t4.append(row)
TABLE4 = pd.DataFrame(_t4)
TABLE4.to_csv(os.path.join(RESULTS_T3_DIR, "table4_standalone_inflation_L2_both_targets.csv"), index=False)
pd.DataFrame({"Leak": TABLE4["leak"],
              "Standalone inflation of mean rho (v1.4, forward)": [_f(v) for v in TABLE4["standalone_inflation_v1_4_forward"]],
              "L2 under the trailing target (v1.2)": [_f(v) if pd.notna(v) else "" for v in TABLE4["standalone_inflation_v1_2_trailing"]],
              "Note": TABLE4["note"]}).to_csv(
    os.path.join(RESULTS_T3_DIR, "table4_standalone_inflation_L2_both_targets_formatted.csv"), index=False)
print("\n(c) Table 4"); print(TABLE4.round(3).to_string(index=False))

# (d) corrected formatted Table 2 (v1.4): "artifact excluded" n/a for V3 vs V-signed
_t2v14 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "task1_table2_bounded_null.csv"), keep_default_na=False, na_values=[""])
for c in ["mean_delta_rho", "ci95_lo", "ci95_hi", "hl_median", "hl_ci95_lo", "hl_ci95_hi", "p", "q", "ci90_lo", "ci90_hi"]:
    _t2v14[c] = pd.to_numeric(_t2v14[c], errors="coerce")
_t2v14.loc[_t2v14["comparison"] == "V3 vs V-signed", "artifact_excluded"] = "n/a"
T2_V14_CORRECTED = format_t2(_t2v14)
T2_V14_CORRECTED.to_csv(os.path.join(RESULTS_T3_DIR, "task1_table2_formatted_v1_4_corrected.csv"), index=False)
print("\n(d) Table 2 (v1.4), corrected"); print(T2_V14_CORRECTED.to_string(index=False))

# (e) reporting notes
REPORTING_NOTES = [
    "REPORTING NOTES (v1.4 review memo)",
    "1. p12 is included in the primary analysis; the without-p12 numbers are a sensitivity analysis. Not robust to "
    "removing p12, and to be written as such: V3 vs V-signed on the self-derived target (P = .018 with p12, .135 "
    "without) and V-signed vs raw-RF (q = .039 with, .055 without).",
    "2. Under the forward target the self-proxy (L2) is a naive persistence forecast, not a leak. Figure 4's "
    "cumulative chain starts from the L2-removed state: " + " -> ".join(f"{v:.3f}" for v in _vals) + ".",
    "3. The raw-feature target is a floor: every arm's mean rho is below 0.10. Say so wherever its null is reported.",
    "4. The equivalence of V3 and V-signed on the raw-feature target is not a finding: both arms are near zero there.",
    "5. 'Artifact excluded' is n/a for V3 vs V-signed (no q) in Table 2.",
    f"6. Target-swap shift without p12 (exploratory, sensitivity): reproduced; matches ma'am's values: {SWAP_NOP12_MATCH}."]
with open(os.path.join(RESULTS_T3_DIR, "reporting_notes_v1_4.txt"), "w") as f:
    f.write("\n".join(REPORTING_NOTES) + "\n")
print("\n" + "\n".join(REPORTING_NOTES))

## Figure 2, independent check, hand-in

In [ ]:
# T3-5 Figure 2 rebuilt as three panels: (A) self-derived and (B) raw-feature targets (results_v1.4, read-only),
# (C) next-day mood (this run, eligible participants). Supplementary figure: next-day stress.
# Dots: one participant each. Black marker and bar: mean rho with BCa 95% CI. No figure number in the image.
import matplotlib.pyplot as plt
_ARM_ORDER = ["vsigned", "v3", "rawrf", "persz"]
_ARM_TICK = ["V-signed", "V3", "Raw-feature\nRF", "Personalised\nZ"]
_ab14 = pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "task1_absolute_rho_both_targets.csv"))
_panels = [("(A) Self-derived target",
            pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "table1_corrected_self_target_per_participant.csv")),
            _ab14[_ab14["target"] == "self-derived"]),
           ("(B) Raw-feature target",
            pd.read_csv(os.path.join(RESULTS_V1_4_ARCHIVE, "raw_target_corrected_per_participant.csv")),
            _ab14[_ab14["target"] == "raw-feature"]),
           ("(C) Next-day self-reported mood", T3_ELIG["mood"], T3_ABS["mood"])]

def _panel(ax, title, pp, ab):
    rng = np.random.default_rng(0)
    for i, a in enumerate(_ARM_ORDER):
        v = pp[a].dropna().values
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=14, alpha=0.55, color="#4C72B0", zorder=2)
        r = ab[ab["arm"] == ARM_LABEL[a]].iloc[0]
        ax.errorbar(i + 0.25, r["mean_rho"], yerr=[[r["mean_rho"] - r["ci95_lo"]], [r["ci95_hi"] - r["mean_rho"]]],
                    fmt="o", color="black", capsize=4, zorder=3)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_xticks(range(4)); ax.set_xticklabels(_ARM_TICK, fontsize=8)
    ax.set_title(f"{title} (n = {len(pp)})", fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, (title, pp, ab) in zip(axes, _panels):
    _panel(ax, title, pp, ab)
axes[0].set_ylabel("Within-participant Spearman ρ")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_T3_DIR, f"figure2_three_panels.{ext}"), dpi=300)
plt.show()
fig, ax = plt.subplots(figsize=(4.8, 4.2))
_panel(ax, "Next-day self-reported stress", T3_ELIG["stress"], T3_ABS["stress"])
ax.set_ylabel("Within-participant Spearman ρ"); fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_T3_DIR, f"figure_supplement_stress.{ext}"), dpi=300)
plt.show()
print("Figure 2 (three panels) and the stress supplement ->", RESULTS_T3_DIR)

In [ ]:
# T3-6 Independent check of the primary Task 3 result (no run_arms, no W-1 functions, no cache).
#   (i)  next-day mood rebuilt from wellness.csv by a separate implementation of W1-W4, row by row,
#        equals the Task 3 target (NaN pattern included), for every participant;
#   (ii) an independent fold loop (own fold rule, own features, own fits) on that target reproduces V3 and raw-RF
#        for every eligible participant (to 1e-9). About 1 minute per participant on Colab.
def x3_next_day(pid, item, dates):
    path = os.path.join(PMDATA_DIR, pid, "pmsys", "wellness.csv")
    if not os.path.isfile(path):
        return np.full(len(dates), np.nan)
    w = pd.read_csv(path); w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
    five = ["mood", "stress", "fatigue", "sleep_quality", "soreness"]
    kept = []
    for order, (_, r) in enumerate(w.iterrows()):
        ts = pd.Timestamp(str(r["effective_time_frame"]).strip())
        vals = [pd.to_numeric(r.get(c), errors="coerce") for c in five]
        if all(v == 0 for v in vals):                       # W3: blank submission, whole report dropped
            continue
        v = pd.to_numeric(r.get(item), errors="coerce")
        if pd.isna(v) or v == 0:                            # W2: 0 = missing on 1-5 items
            continue
        kept.append((ts.tz_convert("UTC"), order, ts.tz_convert("Europe/Oslo").strftime("%Y-%m-%d"), float(v)))  # W1
    last = {}
    for ts, order, day, v in sorted(kept):                  # W4: the last valid value of the day wins
        last[day] = v
    return np.array([last.get((d + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), np.nan) for d in dates])

def x3_fold_rho(p, participants, feat_names, y, g):
    d0 = p["days_df"].copy().reset_index(drop=True); n = len(d0)
    Xn = np.column_stack([pd.to_numeric(d0[c], errors="coerce").values if c in d0.columns else np.zeros(n)
                          for c in feat_names]).astype(float)
    pm, pmad, _ = _compute_population_volatility_norm(participants, p["pid"], feat_names, False)
    t0 = min(10, max(5, n // 3))
    def fill(X, rows):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            med = np.nanmedian(X[rows], axis=0)
        med = np.where(np.isfinite(med), med, 0.0)
        return np.where(np.isnan(X), med, X)
    yt, p3, pr = [], [], []
    for ts in range(t0 + g, n):
        tr = np.arange(0, ts - g)          # next-day target of row t is day t+1 <= ts-g < ts: nothing to purge
        L = ts + 1; tm = np.zeros(L, bool); tm[tr] = True
        X = fill(Xn[:L], tm)
        T = ndi.compute_T_v3(X, tm)
        mi, ii = ndi.compute_quality_signal(d0.iloc[:L].reset_index(drop=True), feat_names, tm)
        I = ndi.compute_I_v3(mi, ii); F = ndi.compute_F_v3(X, feat_names, tm, pm, pmad)
        X3 = np.column_stack([T, I, F])
        trv = tr[np.isfinite(y[tr])]
        if len(trv) < 5 or not np.isfinite(y[ts]):
            continue
        m = ndi.rf_factory(); m.fit(X3[trv], y[trv]); p3.append(float(m.predict(X3[[ts]])[0]))
        m = ndi.rf_factory(); m.fit(X[trv], y[trv]); pr.append(float(m.predict(X[[ts]])[0]))
        yt.append(y[ts])
    return len(yt), spearmanr(yt, p3)[0], spearmanr(yt, pr)[0]

_t = time.time(); _x3 = []
_elig_mood = set(T3_ELIG["mood"]["pid"])
for p in pmdata_participants:
    dates = pd.to_datetime(p["days_df"]["date"]).reset_index(drop=True)
    y_ind = x3_next_day(p["pid"], "mood", dates)
    y_t3 = TASK3_Y[(p["pid"], "mood")]
    same = bool(np.array_equal(np.isnan(y_ind), np.isnan(y_t3)) and np.allclose(y_ind, y_t3, equal_nan=True, atol=0))
    row = dict(pid=p["pid"], target_identical=same, rows_with_target=int(np.isfinite(y_ind).sum()))
    if p["pid"] in _elig_mood:
        nf, r3, rr = x3_fold_rho(p, pmdata_participants, PMDATA_FEATURE_NAMES, y_t3, G)
        ref = T3_ELIG["mood"].set_index("pid").loc[p["pid"]]
        row.update(n_folds_indep=nf, v3_indep=r3, rawrf_indep=rr, n_folds_task3=int(ref["n_folds"]),
                   v3_task3=float(ref["v3"]), rawrf_task3=float(ref["rawrf"]),
                   abs_diff_v3=abs(r3 - ref["v3"]), abs_diff_rawrf=abs(rr - ref["rawrf"]))
    _x3.append(row)
x3 = pd.DataFrame(_x3)
x3.to_csv(os.path.join(RESULTS_T3_DIR, "task3_independent_check.csv"), index=False)
print(x3.round(3).to_string(index=False))
_e = x3.dropna(subset=["n_folds_indep"]) if "n_folds_indep" in x3 else x3.iloc[0:0]
_worst = float(max(_e["abs_diff_v3"].max(), _e["abs_diff_rawrf"].max())) if len(_e) else 0.0
x3_lines = [f"{len(x3)} participants, {time.time() - _t:.0f}s",
            f"(i)  next-day mood rebuilt independently from wellness.csv equals the Task 3 target for "
            f"{int(x3['target_identical'].sum())} of {len(x3)} participants",
            f"(ii) independent fold loop vs Task 3 (V3, raw-RF), {len(_e)} eligible participants: max |diff| = {_worst:.1e}"]
assert x3["target_identical"].all(), "independent mood target differs -- STOP and report"
if len(_e):
    assert (_e["n_folds_indep"] == _e["n_folds_task3"]).all(), "fold counts differ -- STOP and report"
    assert _worst < 1e-9, "independent fold loop does not match Task 3 -- STOP and report"
x3_lines.append("INDEPENDENT CHECK PASSED: Task 3 mood target and V3 / raw-RF reproduced without run_arms or W-1")
print("\n".join(x3_lines))
with open(os.path.join(RESULTS_T3_DIR, "task3_independent_check_notes.txt"), "w") as f:
    f.write("\n".join(x3_lines) + "\n")

In [ ]:
# T3-H Task 3 notes and hand-in check (with the v1.4 review memo additions): the v1.4 reproduction is repeated LAST (about 8 minutes), then files,
# provenance, archives untouched, checks, eligibility.
REPRO_LAST = reproduce_v14("reproduce v1.4 (last)")
_e = T3_ELIGLOG
notes = ["TASK 3 NOTES (revision analyses memo, Task 3; wellness rules W1-W4 from the v1.4 memo)",
         "1. Feature check. " + FEATURE_CHECK,
         "2. Targets. Primary: next-day mood (1-5). Secondary: next-day stress (1-5). Raw next-day value, not smoothed. "
         "Sleep quality, sleep duration, readiness, fatigue and soreness are not used as targets.",
         "3. Day alignment. Each report is dated by its local Europe/Oslo calendar day (W1), the day boundary of the "
         "Fitbit daily files. Row s (Fitbit day d_s) is scored against the W1-W4 value of day d_s + 1. Rows are "
         "consecutive calendar days (D-2).",
         "4. Protocol. Folds, embargo g = 6, fold size 1, features, arms and models as in v1.4; only the target column "
         "changes. With a next-day target the purge removes no training row (target of row t is day t+1 <= ts-6).",
         "5. Missing reports. A test day with no next-day value is dropped from scoring, not imputed; training rows "
         "without a target are dropped from that fold's fit."]
for i, tgt in enumerate(TASK3_TARGETS):
    _t = _e[_e["target"] == tgt]
    notes.append(f"{6 + i}. {tgt}: {int(_t['eligible'].sum())} of {len(_t)} eligible (>= {MF} scored test folds and "
                 f">= 3 distinct values). Excluded: "
                 + ("; ".join(f"{r.pid} -- {r.exclusion_reason}" for r in _t[~_t['eligible']].itertuples()) or "none")
                 + f". Scored test days without a next-day value dropped: {int(_t['test_days_without_next_day_value'].sum())}. "
                 f"W4 chose among 2+ values on {int(_t['scored_days_where_W4_chose_among_2plus_values'].sum())} scored "
                 f"target days; the values differed on {int(_t['scored_days_where_those_values_differed'].sum())}. "
                 f"Analysis: {T3_T2[tgt]['analysis'].iloc[0]}.")
notes += _np_notes
_ps = PERSIST_SUM.set_index("target")
notes.append("Persistence references (no model, today's value; outside the BH set): " + "; ".join(
    f"{t} {_ci(r.mean_rho, r.ci95_lo, r.ci95_hi)} (n = {int(r.participants)}, {int(r.days_used)} days)"
    for t, r in _ps.iterrows()) + f". V3 minus persistence on the self-derived target (exploratory): "
    f"{_ci(V3_VS_PERSIST['mean_delta_rho'].iloc[0], V3_VS_PERSIST['ci95_lo'].iloc[0], V3_VS_PERSIST['ci95_hi'].iloc[0])}, "
    f"P = {_f(V3_VS_PERSIST['p'].iloc[0])}.")
notes.append("Artifact bound: each upper 95% bound is compared with the self-derived advantage of the same eligible "
             "participants (v1.4 per-participant files); n/a for V3 vs V-signed.")
notes.append(f"Sleep-score dating (checked before tagging): {SLEEP_SCORE_CHECK['reason']}; scores without a matching "
             f"log: {SLEEP_SCORE_CHECK['scores_without_matching_log']}; dates with 2+ scores: "
             f"{SLEEP_SCORE_CHECK['cell5_dates_with_2plus_scores']}. v1.4 features unchanged.")
notes.append(f"Checks: v1.4 reproduction first {REPRO_FIRST} and last {REPRO_LAST} (max |diff|); wellness table equals "
             f"v1.4: {WELLNESS_MATCHES_V14}; independent check: {x3_lines[-1]}")
print("\n".join(notes))
with open(os.path.join(RESULTS_T3_DIR, "task3_notes.txt"), "w") as f:
    f.write("\n".join(notes) + "\n")

import datetime as _dt
report, fails = [], []
def say(s=""):
    print(s); report.append(s)
def check(name, ok):
    say(f"   {'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        fails.append(name)
say("\n1. FILES in results_v1.4_task3")
_exp = ["config_task3_v1_4.json", "provenance.json", "task3_eligibility_log.csv", "task3_notes.txt",
        "task3_independent_check.csv", "task3_independent_check_notes.txt", "task3_without_p12_notes.txt",
        "figure2_three_panels.png", "figure2_three_panels.pdf", "figure_supplement_stress.png", "figure_supplement_stress.pdf",
        "persistence_reference_per_participant.csv", "persistence_reference_summary.csv",
        "persistence_reference_summary_formatted.csv", "v3_vs_persistence_self_derived_exploratory.csv",
        "exploratory_target_swap_shift_without_p12.csv", "exploratory_target_swap_shift_without_p12_formatted.csv",
        "figure4_leak_waterfall_from_L2_removed.png", "figure4_leak_waterfall_from_L2_removed.pdf",
        "figure4_from_L2_removed_bar_labels.csv", "table4_standalone_inflation_L2_both_targets.csv",
        "table4_standalone_inflation_L2_both_targets_formatted.csv", "task1_table2_formatted_v1_4_corrected.csv",
        "reporting_notes_v1_4.txt"]
for tgt in TASK3_TARGETS:
    _exp += [f"task3_{tgt}_per_participant.csv", f"task3_fold_log_{tgt}.csv", f"task3_table2_{tgt}.csv",
             f"task3_absolute_rho_{tgt}.csv", f"task3_table2_{tgt}_formatted.csv", f"task3_absolute_rho_{tgt}_formatted.csv"]
    if tgt in T3_NOP12:
        _exp += [f"task3_{tgt}_without_p12_per_participant.csv", f"task3_table2_{tgt}_without_p12.csv",
                 f"task3_absolute_rho_{tgt}_without_p12.csv", f"task3_table2_{tgt}_without_p12_formatted.csv"]
_miss = [f for f in _exp if not os.path.isfile(os.path.join(RESULTS_T3_DIR, f))]
check(f"{len(_exp) - len(_miss)} of {len(_exp)} present" + (f"; MISSING: {_miss}" if _miss else ""), not _miss)
say("2. PROVENANCE")
_prov = json.load(open(os.path.join(RESULTS_T3_DIR, "provenance.json")))
say(f"   run_started {_prov['run_started']} | tag {_prov['tag']} -> {_prov['tag_commit']} | HEAD {_prov['head_commit']}")
check("tag commit recorded and HEAD = tag commit", bool(_prov["tag_commit"]) and _prov["head_commit"] == _prov["tag_commit"])
check("tag commit time precedes run_started", bool(_prov["tag_time_precedes_run_start"]))
check("config used equals the config committed under the tag", bool(_prov["config_matches_tagged_config"]))
check("v1.4 settings read from results_v1.4 equal the config under tag v1.4; F_SCALE = 1.0",
      bool(_prov["v1_4_config_matches_tag"]) and _prov["f_scale"] == 1.0)
check(f"sleep-score check passed before the run ({_prov['sleep_score_check']['mismatches_total']} mismatches, all known)",
      _prov["sleep_score_check"]["verdict"] == "PASS")
say("3. ARCHIVES untouched since this run started")
_start = _dt.datetime.fromisoformat(_prov["run_started"]).timestamp()
for _n in ("results_v1.2", "results_v1.3", "results_v1.4", "results_final_v1_frozen_backup"):
    _d = os.path.join(_BASE_DIR, _n)
    if os.path.isdir(_d):
        _late = [f for f in os.listdir(_d) if os.path.getmtime(os.path.join(_d, f)) >= _start]
        check(f"{_n}: no file modified ({len(os.listdir(_d))} files)", not _late)
    else:
        check(f"{_n} folder found", False)
say("4. CHECKS")
check(f"v1.4 reproduced exactly, first {REPRO_FIRST} and last {REPRO_LAST}",
      max(REPRO_FIRST.values()) < 1e-9 and max(REPRO_LAST.values()) < 1e-9)
check("wellness table identical to results_v1.4/wellness_daily_W1_W4.csv", WELLNESS_MATCHES_V14)
check("independent check of the mood target and of V3 / raw-RF passed", x3_lines[-1].startswith("INDEPENDENT CHECK PASSED"))
check("no wellness.csv field in any arm's features", not (set(PMDATA_FEATURE_NAMES) & _well_fields))
check("every eligible participant has a finite rho in all four arms (asserted in T3-2 and T3-4)", True)
check(f"persistence on the self-derived target equals v1.4 'Clean + L2 only' ({L2_ARCHIVED:.3f})", L2_MATCH)
check("today's raw composite rebuilds the v1.4 raw-feature target exactly", RAW_COMPOSITE_MATCH)
check("target-swap shift without p12 matches ma'am's values (mean and q, 3 decimals)", SWAP_NOP12_MATCH)
check("next-day purge removed no training row", all(int(RUN_LOGS[f"task3 {t}"]["purged_rows_summed_over_folds"].sum()) == 0
                                                     for t in TASK3_TARGETS))
say("5. ELIGIBILITY")
for tgt in TASK3_TARGETS:
    say(f"   {tgt}: {len(T3_ELIG[tgt])} eligible -> {T3_T2[tgt]['analysis'].iloc[0]}")
say("")
say("TASK 3 HAND-IN CHECK " + ("PASSED: every check above passed" if not fails else f"FAILED: {fails}"))
with open(os.path.join(RESULTS_T3_DIR, "handin_check_task3.txt"), "w") as f:
    f.write("\n".join(report) + "\n")